# Once Upon a Matchmaker, in Python: the optional notebook

**This notebook is optional.** It holds parts 3 and 4, which go with the optional game of the Matchmaker,
gradient descent up close. Parts 1 and 2 are in [the main notebook](https://colab.research.google.com/github/eth-ainit-hs26/w2-cx-public/blob/main/02_cx_matrix_factorization_solutions.ipynb). Do each part when the game sends you
here, after its half:

3. **Part 3, gradient descent**, after the first half, "Smaller hops": feel the slope with a hop of 0.01, then hop
   downhill.
4. **Part 4, many customers, many films**, after the second half, "Many customers, many films": the average loss
   of a whole table, the slope of every number, and the loop `factorize` that finds both matrices from the
   ratings alone.

Each part stands on its own: it gives again what it needs from parts 1 and 2.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A
folded cell with a title holds plumbing: run it, and open it only if you are curious. The 💡 hints are folded
cells too: double-click one to open it, and open the next one only if you are still stuck. After each task, a
check cell tells you whether your code works.

# Part 3. Smaller hops: gradient descent

In part two of the game the Matchmaker moved each number by an amount it computed out of sight. The first half
of the optional game shows how: a grasshopper hops a tiny step, 0.01, feels how steep the ground is, the
**slope**, and then jumps downhill by a small amount, **alpha** times the slope. Done again and again, that is
**gradient descent**. In the code the size of the hop is called `eps`.

You teach the computer the grasshopper's moves on one rating, the corner of the table, Ida and *Paddington*:

- **Task A**: the slope from a tiny hop;
- **Task B**: the slope from a formula;
- **Task C**: one step downhill for all four guessed numbers at once;
- **Task D**: gradient descent, step after step, until the loss is about 0.

Then five experiments. Part 3 stands on its own: it gives again what it needs from parts 1 and 2. Run its setup
cell first.

In [ ]:
#@title Setup of part 3 (run this first) { display-mode: "form" }
# The plumbing of part 3: show_corner, which draws the corner the way the game does, print_hop, print_slopes and
# print_step, which print results in words, show_losses, which prints the list of losses of gradient descent, and
# the checks of tasks A to D. You do not need to read it. A check prints one short message and never stops the
# notebook.

# Part 3 uses inner_product from part 1, which is in the main notebook. This cell defines it when it is missing,
# so that part 3 works on its own. The loss comes in the cell after the setup.
if "inner_product" not in globals():

    def inner_product(row_a, row_b):
        # row_a and row_b are two lists of numbers of the same length,
        # such as a customer's row and a film's column.
        # Multiply them position by position, then add up the products.
        total = 0
        for i in range(len(row_a)):
            total = total + row_a[i] * row_b[i]
        return total

# Whose numbers: the row's (0) and the column's (1). The rating of the corner. The game's hop, then the two other
# hops the check of task A tries. Below _P3_TOL the loss counts as 0. Two decimal numbers count as equal within the
# relative margin _P3_MARGIN. The check of task D stops a loop that makes more than _P3_MAX_CALLS steps.
_P3_OWNERS = ['Ida', 'Paddington']
_P3_POSITION_NAMES = ["first", "second"]
_P3_RATING = 2
_P3_HOPS = [0.01, 0.1, 0.001]
_P3_TOL = 0.0001
_P3_MARGIN = 1e-9
_P3_MAX_CALLS = 5000
_P3_ORDER = [(0, 0), (0, 1), (1, 0), (1, 1)]      # (whose, position): the row's two numbers, then the column's


def _p3_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 80:
        text = text[:77] + "..."
    return text


def _p3_is_number(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


def _p3_dec(x, places):
    """x rounded to at most places decimals, without trailing zeros: -5.91, 0.0591, 4."""
    if not _p3_is_number(x):
        return _p3_short(x)
    if x != x or x == float("inf") or x == float("-inf"):
        return str(x)
    text = "%.*f" % (places, x)
    if "." in text:
        text = text.rstrip("0").rstrip(".")
    if text == "-0":
        text = "0"
    return text


def _p3_loss_text(x):
    """A loss as text: at most 4 decimals, two digits for a small one, and about 0 below 0.0001."""
    if _p3_is_number(x) and 0 <= x < _P3_TOL:
        return "about 0"
    if _p3_is_number(x) and 0 < x < 0.001:
        return _p3_dec(x, 5)
    return _p3_dec(x, 4)


def _p3_name(whose, position):
    """Ida's first number"""
    return "%s's %s number" % (_P3_OWNERS[whose], _P3_POSITION_NAMES[position])


_P3_NAME_WIDTH = max(len(_p3_name(w, p)) for w, p in _P3_ORDER) + 1


def _p3_count(n, word):
    """1 step, 4 steps"""
    if n == 1:
        return "1 " + word
    return "%d %ss" % (n, word)


def _p3_pair(value):
    """True when value is a list of two numbers, the shape of the row and of the column of the corner."""
    if not isinstance(value, list) or len(value) != 2:
        return False
    for x in value:
        if not _p3_is_number(x):
            return False
    return True


def _p3_waiting(value):
    """Why a value is not a number yet, in words."""
    if value is Ellipsis:
        return "it is still ..., a gap in a 🎯 cell"
    if value is None:
        return "it is None (does the function end with a return line?)"
    return "it is %s" % _p3_short(value)


# The helpers that draw the corner and print results in words

def show_corner(row, column):
    """The corner as the game draws it: the row on the left, the column above, and where they meet the prediction,
    the inner product of the two, with decimals rounded to 2 places. For the stuck guess:
            Paddington
                    -3
                     2
    Ida   -3   -3    3   the prediction"""
    try:
        for value, which in ((row, "the row"), (column, "the column")):
            if not _p3_pair(value):
                print("⏳ Nothing to show: %s is %s, and it should be a list of two numbers."
                      % (which, _p3_short(value)))
                return
        prediction = _p3_inner(row, column)
        texts = []
        for x in row + column + [prediction]:
            texts.append(_p3_dec(x, 2))
        width = max(len(text) for text in texts)
        left = len(_P3_OWNERS[0]) + 3 + width + 3 + width + 3       # where the column and the prediction start
        print(" " * max(0, left + width - len(_P3_OWNERS[1])) + _P3_OWNERS[1])
        for x in column:
            print(" " * left + _p3_dec(x, 2).rjust(width))
        print("%s   %s   %s   %s   the prediction" % (_P3_OWNERS[0], _p3_dec(row[0], 2).rjust(width),
                                                     _p3_dec(row[1], 2).rjust(width),
                                                     _p3_dec(prediction, 2).rjust(width)))
    except Exception as error:
        print("The corner could not be shown (%s)." % error)


def print_hop(eps, slope_found):
    """One line of the grasshopper's table: the hop and the slope it measured, at most 4 decimals."""
    try:
        if not _p3_is_number(slope_found):
            print("A hop of %s: ⏳ no slope yet, %s. Finish task A, run its cell, then run this cell again."
                  % (_p3_dec(eps, 6), _p3_waiting(slope_found)))
            return
        print("A hop of %s gives the slope %s" % (_p3_dec(eps, 6).ljust(6), _p3_dec(slope_found, 4)))
    except Exception:
        pass


def print_slopes(owner, position, from_formula, from_hop):
    """One line of the table of slopes: whose number (a name) and its position, its slope from the formula and its
    slope from a tiny hop."""
    try:
        name = ("%s's %s number:" % (owner, _P3_POSITION_NAMES[position])).ljust(_P3_NAME_WIDTH)
        waiting = []
        if not _p3_is_number(from_formula):
            waiting.append("slope, task B")
        if not _p3_is_number(from_hop):
            waiting.append("slope_by_hop, task A")
        if waiting:
            print("%s ⏳ not finished yet: %s" % (name, " and ".join(waiting)))
            return
        print("%s the slope %s by a tiny hop %s" % (name, _p3_dec(from_formula, 4).ljust(6), _p3_dec(from_hop, 4)))
    except Exception:
        pass


def print_step(step_number, row, column, loss):
    """One step of gradient descent in words, for example
    Step  1:  the loss 0.0553    the prediction 1.7648    Ida -2.88 and -3.08, Paddington -2.88 and 2.12
    It prints nothing while the loss is still ..., a gap in the 🎯 cell of task D, and one calm line at step 1
    when the loss is something else."""
    try:
        if not _p3_is_number(loss):
            if loss is not Ellipsis and step_number == 1:
                print("Step  1:  ⏳ no loss yet: %s. The checks of tasks C and D say more." % _p3_waiting(loss))
            return
        text = "Step %2s:  the loss %s" % (step_number, _p3_loss_text(loss).ljust(8))
        if _p3_pair(row) and _p3_pair(column):
            text = text + "  the prediction %s  %s %s and %s, %s %s and %s" % (
                _p3_dec(_p3_inner(row, column), 4).ljust(8),
                _P3_OWNERS[0], _p3_dec(row[0], 2), _p3_dec(row[1], 2),
                _P3_OWNERS[1], _p3_dec(column[0], 2), _p3_dec(column[1], 2))
        print(text.rstrip())
    except Exception:
        pass


def show_losses(losses):
    """The list of losses that gradient_descent gives back, in one line: the number of steps, then the losses,
    the loss at the start first (all of them, or the first five and the last three)."""
    try:
        if losses is Ellipsis or losses is None or not isinstance(losses, list) or len(losses) == 0:
            if isinstance(losses, list):
                reason = "the list of losses is empty"
            else:
                reason = "gradient_descent gave no list yet: " + _p3_waiting(losses)
            print("⏳ Nothing to show yet: %s. Finish task D, run its cell, then run this cell again." % reason)
            return
        for x in losses:
            if not _p3_is_number(x):
                print("⏳ The list of losses still holds %s. The check of task D says more." % _p3_short(x))
                return
        texts = []
        for x in losses:
            texts.append(_p3_loss_text(x))
        if len(texts) > 9:
            texts = texts[:5] + ["..."] + texts[-3:]
        steps = len(losses) - 1
        if steps == 0:
            print("No step. The loss: %s" % texts[0])
        else:
            print("%s. The losses: %s" % (_p3_count(steps, "step"), ", ".join(texts)))
    except Exception as error:
        print("The losses could not be shown (%s)." % error)


# The arithmetic, kept here for the checks

def _p3_inner(row_a, row_b):
    total = 0
    for i in range(len(row_a)):
        total = total + row_a[i] * row_b[i]
    return total


def _p3_loss(row, column, rating):
    """The loss of the corner: the prediction minus the rating, squared. The two lists may come in either order."""
    difference = _p3_inner(row, column) - rating
    return difference * difference


def _p3_with(numbers, position, amount):
    """A copy of numbers with amount added to numbers[position]."""
    out = list(numbers)
    out[position] = out[position] + amount
    return out


def _p3_slope(numbers, partners, rating, position):
    """The slope at numbers[position]: 2 times its partner times the prediction minus the rating."""
    difference = _p3_inner(numbers, partners) - rating
    return 2 * partners[position] * difference


def _p3_step(row, column, rating, alpha):
    """One step of gradient descent, in place, as gradient_step should do it. Gives back the loss after it."""
    row_slopes = [0, 0]
    column_slopes = [0, 0]
    for position in range(2):
        row_slopes[position] = _p3_slope(row, column, rating, position)
        column_slopes[position] = _p3_slope(column, row, rating, position)
    for position in range(2):
        row[position] = row[position] - alpha * row_slopes[position]
        column[position] = column[position] - alpha * column_slopes[position]
    return _p3_loss(row, column, rating)


def _p3_step_variant(row, column, rating, alpha, how):
    """[the row, the column] after one step done with a slip: how is "one at a time", "row first",
    "row, then column", "no swap", "plus", "no alpha" or "divided"."""
    r = list(row)
    c = list(column)
    if how == "one at a time":                      # each number moves as soon as its slope is measured
        for position in range(2):
            r[position] = r[position] - alpha * _p3_slope(r, c, rating, position)
            c[position] = c[position] - alpha * _p3_slope(c, r, rating, position)
        return [r, c]
    if how == "row first":                          # the same, the row's two numbers before the column's
        for position in range(2):
            r[position] = r[position] - alpha * _p3_slope(r, c, rating, position)
        for position in range(2):
            c[position] = c[position] - alpha * _p3_slope(c, r, rating, position)
        return [r, c]
    if how == "row, then column":                   # the row's slopes, the row moves, then the column's slopes
        row_slopes = [_p3_slope(r, c, rating, position) for position in range(2)]
        for position in range(2):
            r[position] = r[position] - alpha * row_slopes[position]
        column_slopes = [_p3_slope(c, r, rating, position) for position in range(2)]
        for position in range(2):
            c[position] = c[position] - alpha * column_slopes[position]
        return [r, c]
    row_slopes = [0, 0]
    column_slopes = [0, 0]
    for position in range(2):
        row_slopes[position] = _p3_slope(r, c, rating, position)
        if how == "no swap":                        # the column's slopes measured with the row first
            column_slopes[position] = _p3_slope(r, c, rating, position)
        else:
            column_slopes[position] = _p3_slope(c, r, rating, position)
    for position in range(2):
        for numbers, slopes in ((r, row_slopes), (c, column_slopes)):
            if how == "plus":
                numbers[position] = numbers[position] + alpha * slopes[position]
            elif how == "no alpha":
                numbers[position] = numbers[position] - slopes[position]
            elif how == "divided":
                numbers[position] = numbers[position] - slopes[position] / alpha
            else:
                numbers[position] = numbers[position] - alpha * slopes[position]
    return [r, c]


# The checks

# Each test: the inputs, the expected answer, the call as you would type it, and why that is the answer.
_P3_TESTS = {
    "slope_by_hop": [
        (([-3, -3], [-3, 2], 2, 0, 0.01), -5.9099999999998705,
         'slope_by_hop([-3, -3], [-3, 2], 2, 0, 0.01)',
         "The game's hop: the loss goes from 1 to 0.9409, a change of -0.0591, and -0.0591 / 0.01 is -5.91"),
        (([-3, -3], [-3, 2], 2, 0, 0.1), -5.100000000000009,
         'slope_by_hop([-3, -3], [-3, 2], 2, 0, 0.1)',
         'Another hop, 0.1: the loss goes from 1 to 0.49, and -0.51 / 0.1 is -5.1'),
        (([-3, -3], [-3, 2], 2, 0, 0.001), -5.991000000000191,
         'slope_by_hop([-3, -3], [-3, 2], 2, 0, 0.001)',
         'Another hop, 0.001: -0.005991 / 0.001 is -5.991'),
        (([-3, 2], [-3, -3], 2, 0, 0.01), -5.9099999999998705,
         'slope_by_hop([-3, 2], [-3, -3], 2, 0, 0.01)',
         "Paddington's first number: its partner is Ida's first, -3, so it gets the same slope as Ida's first number"),
        (([-3, -3], [-3, 2], 2, 1, 0.01), 4.03999999999991,
         'slope_by_hop([-3, -3], [-3, 2], 2, 1, 0.01)',
         "Ida's second number: its partner is Paddington's second, 2, and the slope by this hop is 4.04"),
        (([-1, 2], [-1, -2], 2, 1, 0.01), 20.03999999999948,
         'slope_by_hop([-1, 2], [-1, -2], 2, 1, 0.01)',
         "Part 2's first guess, Paddington's second number: the slope by this hop is 20.04"),
        (([1, 1], [1, 1], -1, 0, 0.01), 6.009999999999849,
         'slope_by_hop([1, 1], [1, 1], -1, 0, 0.01)',
         'Another rating, -1: the slope by this hop is 6.01'),
        (([-2.88, 2.12], [-2.88, -3.08], 2, 1, 0.01), 1.5436959999999667,
         'slope_by_hop([-2.88, 2.12], [-2.88, -3.08], 2, 1, 0.01)',
         'A guess with decimals, one step downhill from the stuck guess: the slope by this hop is 1.5437'),
    ],
    "slope": [
        (([-3, -3], [-3, 2], 2, 0), -6,
         'slope([-3, -3], [-3, 2], 2, 0)',
         "The stuck guess, Ida's first number: 2 times its partner (-3) times the prediction minus the rating (1) is -6"),
        (([-3, -3], [-3, 2], 2, 1), 4,
         'slope([-3, -3], [-3, 2], 2, 1)',
         "The stuck guess, Ida's second number: 2 times its partner (2) times the prediction minus the rating (1) is 4"),
        (([-3, 2], [-3, -3], 2, 0), -6,
         'slope([-3, 2], [-3, -3], 2, 0)',
         "The stuck guess, Paddington's first number: 2 times its partner (-3) times the prediction minus the rating (1) is -6"),
        (([-3, 2], [-3, -3], 2, 1), -6,
         'slope([-3, 2], [-3, -3], 2, 1)',
         "The stuck guess, Paddington's second number: 2 times its partner (-3) times the prediction minus the rating (1) is -6"),
        (([-1, -2], [-1, 2], 2, 0), 10,
         'slope([-1, -2], [-1, 2], 2, 0)',
         "Part 2's first guess, Ida's first number: 2 times its partner (-1) times the prediction minus the rating (-5) is 10"),
        (([-1, -2], [-1, 2], 2, 1), -20,
         'slope([-1, -2], [-1, 2], 2, 1)',
         "Part 2's first guess, Ida's second number: 2 times its partner (2) times the prediction minus the rating (-5) is -20"),
        (([-1, 2], [-1, -2], 2, 0), 10,
         'slope([-1, 2], [-1, -2], 2, 0)',
         "Part 2's first guess, Paddington's first number: 2 times its partner (-1) times the prediction minus the rating (-5) is 10"),
        (([-1, 2], [-1, -2], 2, 1), 20,
         'slope([-1, 2], [-1, -2], 2, 1)',
         "Part 2's first guess, Paddington's second number: 2 times its partner (-2) times the prediction minus the rating (-5) is 20"),
        (([1, 1], [1, 1], -1, 1), 6,
         'slope([1, 1], [1, 1], -1, 1)',
         "Another rating, -1, Paddington's second number: 2 times its partner (1) times the prediction minus the rating (3) is 6"),
        (([-2.88, -3.08], [-2.88, 2.12], 2, 1), -0.9972480000000032,
         'slope([-2.88, -3.08], [-2.88, 2.12], 2, 1)',
         "A guess with decimals, Ida's second number: 2 times its partner (2.12) times the prediction minus the rating (-0.2352) is -0.997248"),
        (([-2, -1], [-2, 2], 2, 0), 0,
         'slope([-2, -1], [-2, 2], 2, 0)',
         'This guess gives the rating exactly: the prediction minus the rating is 0, so every slope is 0, the bottom of the valley'),
        (([0, 0], [0, 0], 2, 0), 0,
         'slope([0, 0], [0, 0], 2, 0)',
         'All four at 0: the partner is 0, so the slope is 0'),
    ],
    "gradient_step": [
        (([-3, -3], [-3, 2], 2, 0.02), (0.05531904000000035, [[-2.88, -3.08], [-2.88, 2.12]]),
         'gradient_step([-3, -3], [-3, 2], 2, 0.02)',
         'One step from the stuck guess: the slopes -6, 4, -6 and -6, so Ida -2.88 and -3.08, Paddington -2.88 and 2.12, and the loss 0.0553'),
        (([-1, -2], [-1, 2], 2, 0.02), (9.734400000000003, [[-1.2, -1.6], [-1.2, 1.6]]),
         'gradient_step([-1, -2], [-1, 2], 2, 0.02)',
         "Part 2's first guess: the slopes 10, -20, 10 and 20, so Ida -1.2 and -1.6, Paddington -1.2 and 1.6, and the loss 9.7344"),
        (([-3, -3], [-3, 2], 2, 0.1), (25.8064, [[-2.4, -3.4], [-2.4, 2.6]]),
         'gradient_step([-3, -3], [-3, 2], 2, 0.1)',
         'A big step, alpha 0.1: Ida -2.4 and -3.4, Paddington -2.4 and 2.6, and the loss jumps to 25.8064'),
        (([0, 0], [0, 0], 2, 0.02), (4.0, [[0.0, 0.0], [0.0, 0.0]]),
         'gradient_step([0, 0], [0, 0], 2, 0.02)',
         'All four at 0: every slope is 0, so no number moves, and the loss stays 4'),
        (([1, 1], [1, 1], -1, 0.05), (3.9204, [[0.7, 0.7], [0.7, 0.7]]),
         'gradient_step([1, 1], [1, 1], -1, 0.05)',
         'Another rating, -1, and another step size, 0.05: every slope is 6, so every number becomes 0.7'),
        (([-2.88, -3.08], [-2.88, 2.12], 2, 0.02), (0.0027620280320285163, [[-2.90709504, -3.06005504], [-2.90709504, 2.09102336]]),
         'gradient_step([-2.88, -3.08], [-2.88, 2.12], 2, 0.02)',
         'The second step from the stuck guess: the loss falls from 0.0553 to 0.0028'),
    ],
    "gradient_descent": [
        (([-3, -3], [-3, 2], 2, 0.02, 100), ([1, 0.05531904000000035, 0.0027620280320285163, 0.00014029558408464734, 7.096244225543689e-06], [[-2.9023581895511557, -3.063457044455762], [-2.9023581895511557, 2.0960043146053766]]),
         'gradient_descent([-3, -3], [-3, 2], 2, 0.02, 100)',
         'From the stuck guess the loss is below 0.0001 after 4 steps: 1, 0.0553, 0.0028, 0.00014, about 0'),
        (([-1, -2], [-1, 2], 2, 0.02, 100), ([25, 9.734400000000003, 4.575509425594344, 2.2282869381221087, 1.0789733931627243, 0.5130212150797621, 0.23914170798084858, 0.10957071728443307, 0.04953024234347733, 0.022167571221124623, 0.009851186384560104, 0.0043563619484046695, 0.0019200021176837222, 0.0008442960439546177, 0.0003707039893276124, 0.0001625993225817433, 7.12716215423046e-05], [[-1.7631362247403484, -1.0569255409688034], [-1.7631362247403484, 1.0569255409688034]]),
         'gradient_descent([-1, -2], [-1, 2], 2, 0.02, 100)',
         "From part 2's first guess the loss is below 0.0001 after 16 steps"),
        (([0, 0], [0, 0], 2, 0.02, 5), ([4, 4.0, 4.0, 4.0, 4.0, 4.0], [[0.0, 0.0], [0.0, 0.0]]),
         'gradient_descent([0, 0], [0, 0], 2, 0.02, 5)',
         'All four at 0: no number moves, the loss stays 4, and the loop makes all 5 steps'),
        (([-1, -2], [-1, 2], 2, 0.02, 3), ([25, 9.734400000000003, 4.575509425594344, 2.2282869381221087], [[-1.46524784359637, -1.2805061537274265], [-1.46524784359637, 1.2805061537274265]]),
         'gradient_descent([-1, -2], [-1, 2], 2, 0.02, 3)',
         'At most 3 steps: after them the loss, 2.2283, is still above 0.0001, so the list holds 4 losses'),
        (([-3, -3], [-3, 2], 2, 0.1, 3), ([1, 25.8064, 486.5594015046958, 77270.00315614797], [[16.5067740020736, 3.0108866789375983], [16.5067740020736, 2.491371321753598]]),
         'gradient_descent([-3, -3], [-3, 2], 2, 0.1, 3)',
         'A step that is too big, alpha 0.1: the losses grow, 1, 25.8064, 486.5594, 77270.0032'),
        (([-1, -2], [-1, 2], -1, 0.02, 100), ([4, 1.4864486399999997, 0.6091710259532649, 0.26096153149821866, 0.11439243525280497, 0.05079540781725127, 0.02272863107825644, 0.010218013680397781, 0.004607365352144285, 0.0020814814769897837, 0.0009415370345153641, 0.00042624750877792494, 0.00019307459128392227, 8.748784497057738e-05], [[-1.2400480202706419, -1.5959550702660323], [-1.2400480202706419, 1.5959550702660323]]),
         'gradient_descent([-1, -2], [-1, 2], -1, 0.02, 100)',
         'Another rating, -1: the loss is below 0.0001 after 13 steps'),
        (([1, 2], [-1, -2], 2, 0.02, 10), ([49, 21.086463999999996, 13.892412417573809, 10.566486347751631, 8.68207307309436, 7.488784066914709, 6.6777660740884865, 6.099019311058385, 5.671159068123074, 5.346337729107161, 5.094649343766063], [[0.22677433366338107, 0.45354866732676213], [-0.22677433366338107, -0.45354866732676213]]),
         'gradient_descent([1, 2], [-1, -2], 2, 0.02, 10)',
         'A flipped guess: the loss falls towards 4 and never gets below 0.0001, so the loop makes all 10 steps'),
    ],
}

_P3_SUCCESS = {'slope_by_hop': "The game's hop of 0.01 gives -5.91, and both lists stay as they were.", 'slope': 'At the stuck guess the four slopes are -6, 4, -6 and -6.', 'gradient_step': 'From the stuck guess one step gives Ida -2.88 and -3.08, Paddington -2.88 and 2.12, and the loss 0.0553.', 'gradient_descent': 'From the stuck guess the loss is below 0.0001 after 4 steps.'}


class _P3_Stop(Exception):
    """Raised by the recorder of check_gradient_descent to stop a call that cannot go on."""


def _p3_call(fn, inputs):
    """Run a participant's function on fresh copies of the inputs, with its prints silenced. Gives back (answer,
    error, the copies after the call); error is None when the call went through. Never raises."""
    import contextlib, copy, io
    copies = copy.deepcopy(inputs)
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*copies)
        return answer, None, copies
    except _P3_Stop as stop:
        return None, "STOP " + str(stop), copies
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error), copies


def _p3_close(a, b):
    """True when a and b are numbers that agree within the margin (relative, and absolute near 0)."""
    if not _p3_is_number(a) or not _p3_is_number(b):
        return False
    try:
        return abs(a - b) <= _P3_MARGIN * max(1.0, abs(b))
    except Exception:
        return False


def _p3_close_pair(a, b):
    """True when a and b are both lists of two numbers that agree within the margin."""
    if not _p3_pair(a) or not _p3_pair(b):
        return False
    return _p3_close(a[0], b[0]) and _p3_close(a[1], b[1])


def _p3_close_corner(a, b):
    """True when a and b are both [a row, a column] that agree within the margin."""
    if not isinstance(a, list) or not isinstance(b, list) or len(a) != 2 or len(b) != 2:
        return False
    return _p3_close_pair(a[0], b[0]) and _p3_close_pair(a[1], b[1])


def _p3_close_list(a, b):
    """True when a and b are lists of numbers of the same length that agree within the margin."""
    if not isinstance(a, list) or not isinstance(b, list) or len(a) != len(b):
        return False
    for k in range(len(a)):
        if not _p3_close(a[k], b[k]):
            return False
    return True


def _p3_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    # A gap such as "x = ..." stays in the compiled function; a gap that stands on its own line is only
    # visible in the source, which Jupyter and Colab keep for every cell
    code = getattr(fn, "__code__", None)
    if code is not None and Ellipsis in code.co_consts:
        return True
    import inspect
    try:
        lines = inspect.getsource(fn).splitlines()
    except Exception:
        return False
    for line in lines:
        if "..." in line.split("#")[0]:
            return True
    return False


def _p3_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _p3_pair_text(values):
    """[-2.88, -3.08], with at most 4 decimals."""
    if not _p3_pair(values):
        return _p3_short(values)
    return "[%s, %s]" % (_p3_dec(values[0], 4), _p3_dec(values[1], 4))


def _p3_corner_text(corner):
    """the row [-2.88, -3.08] and the column [-2.88, 2.12]"""
    return "the row %s and the column %s" % (_p3_pair_text(corner[0]), _p3_pair_text(corner[1]))


def _p3_list_text(values):
    """[1, 0.0553, 0.0028, ...], with at most 6 decimals, cut short when it is long."""
    if not isinstance(values, list):
        return _p3_short(values)
    texts = []
    for x in values:
        if _p3_is_number(x):
            texts.append(_p3_dec(x, 6))
        else:
            texts.append(_p3_short(x))
    if len(texts) > 9:
        texts = texts[:5] + ["..."] + texts[-3:]
    return "[" + ", ".join(texts) + "]"


def _p3_error_hint(error, name):
    """A hint for an error that stopped a participant's function."""
    if "UnboundLocalError" in error or "before assignment" in error or "not callable" in error:
        return ("a variable and a function share a name. Inside a function, slope = ... makes slope a number, "
                "and Python forgets the function slope there. Give the variable another name, for example s. "
                "If you typed slope = ... in a cell, run the cell of task B again to bring the function back.")
    if "NameError" in error:
        return ("Python does not know a name that %s uses. Run the setup cell of part 3, the cell that gives "
                "corner_loss, and the cells of the tasks above, then the cell of %s, then this check."
                % (name, name))
    if "ZeroDivisionError" in error:
        return "divide by eps, the size of the hop, and only by eps."
    if "TypeError" in error and "positional argument" in error:
        return ("a function got a different number of things in its brackets than it takes. Keep the first line "
                "of each 🎯 cell as it is given: slope_by_hop(numbers, partners, rating, position, eps), "
                "slope(numbers, partners, rating, position), gradient_step(row, column, rating, alpha).")
    if "IndexError" in error:
        if name in ("slope_by_hop", "slope"):
            return ("numbers and partners are lists of two numbers, so a position is 0 or 1: numbers[position] is "
                    "the number, and partners[position] is its partner.")
        return "row and column are lists of two numbers, so a position is 0 or 1: row[position], column[position]."
    if name == "slope_by_hop":
        return ("corner_loss(numbers, partners, rating) takes the two lists and the rating, and numbers[position] "
                "is the one number that hops.")
    if name == "slope":
        return ("inner_product(numbers, partners) takes the two lists; the partner is one number, "
                "partners[position].")
    if name == "gradient_step":
        return ("slope takes the list that holds the number first, then the other list, the rating and the "
                "position: slope(row, column, rating, position) for a number of the row, slope(column, row, "
                "rating, position) for a number of the column. It gives back one number.")
    return ("gradient_step(row, column, rating, alpha) gives back one number, the loss after the step: loss = "
            "gradient_step(row, column, rating, alpha), then losses.append(loss).")


def _p3_slip(fn, name, slips):
    """The hint of the first slip whose answers fn gives on every test of name, or None."""
    for slip, hint in slips:
        same = True
        for inputs, expected, call, why in _P3_TESTS[name]:
            answer, error, after = _p3_call(fn, inputs)
            try:
                wanted = slip(*inputs)
            except Exception:
                same = False
                break
            if error is not None or not _p3_close(answer, wanted):
                same = False
                break
        if same:
            return hint
    return None


# Task A, slope_by_hop

def _p3_hop_slips():
    L = _p3_loss
    W = _p3_with
    return [
        (lambda n, q, r, p, eps: L(W(n, p, eps), q, r) - L(n, q, r),
         "this is the change of the loss. The slope is the change per unit of hop: return change / eps."),
        (lambda n, q, r, p, eps: (L(n, q, r) - L(W(n, p, eps), q, r)) / eps,
         "the sign is flipped. The change is the loss after the hop minus the loss before: change = loss_after - "
         "loss_before."),
        (lambda n, q, r, p, eps: L(n, q, r) - L(W(n, p, eps), q, r),
         "the change is the loss after the hop minus the loss before, loss_after - loss_before, and the slope is "
         "that change divided by eps."),
        (lambda n, q, r, p, eps: (L(W(n, p, eps), q, r) - L(n, q, r)) * eps,
         "the change was multiplied by eps. The slope is the change per unit of hop: divide, change / eps."),
        (lambda n, q, r, p, eps: (L(W(n, p, 1), q, r) - L(n, q, r)) / eps,
         "the number hopped by 1, a whole hop. The grasshopper hops by eps: numbers[position] = original + eps."),
        (lambda n, q, r, p, eps: L(W(n, p, 1), q, r) - L(n, q, r),
         "the number hopped by 1, a whole hop. The grasshopper hops by eps, original + eps, and the slope is "
         "the change divided by eps."),
        (lambda n, q, r, p, eps: (L(n, W(q, p, eps), r) - L(n, q, r)) / eps,
         "your code hopped the partner, partners[position]. The number that hops is numbers[position], in the "
         "first list."),
        (lambda n, q, r, p, eps: (L(W(n, 0, eps), q, r) - L(n, q, r)) / eps,
         "the number that hops is numbers[position]: position says which of the two numbers it is. Your code "
         "always hopped numbers[0], the first."),
        (lambda n, q, r, p, eps: (L(W(n, p, eps), q, r) - L(n, q, r)) / _P3_HOPS[0],
         "divide by eps, the hop that the function gets, in place of %s. The check also tries the hops %s and %s."
         % (_P3_HOPS[0], _P3_HOPS[1], _P3_HOPS[2])),
        (lambda n, q, r, p, eps: (L(W(n, p, _P3_HOPS[0]), q, r) - L(n, q, r)) / eps,
         "hop by eps, the hop that the function gets: original + eps, in place of %s. The check also tries the hops "
         "%s and %s." % (_P3_HOPS[0], _P3_HOPS[1], _P3_HOPS[2])),
        (lambda n, q, r, p, eps: 0,
         "the loss after the hop came out equal to the loss before. Measure it after the line that hops: "
         "loss_after = corner_loss(numbers, partners, rating), right below numbers[position] = original + eps."),
        (lambda n, q, r, p, eps: L(W(n, p, eps), q, r),
         "this is the loss after the hop. Return the change per unit of hop: (loss_after - loss_before) / eps."),
    ]


def _p3_run_slope_by_hop(fn):
    """None when slope_by_hop passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P3_TESTS["slope_by_hop"]:
        answer, error, copies = _p3_call(fn, inputs)
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p3_error_hint(error, "slope_by_hop"))
        if _p3_close(answer, expected):
            if not _p3_close_pair(copies[0], inputs[0]) or not _p3_close_pair(copies[1], inputs[1]):
                return ("%s gave the right slope, %s, but left the lists at %s and %s; they should stay %s and %s"
                        % (call, _p3_dec(expected, 6), _p3_pair_text(copies[0]), _p3_pair_text(copies[1]),
                           _p3_pair_text(inputs[0]), _p3_pair_text(inputs[1])),
                        "the grasshopper only feels the ground and hops back: after the loss after the hop is "
                        "measured, put the original back, numbers[position] = original.")
            continue
        if answer is None:
            return ("%s gave no answer (None). Does the function end with a return line?" % call,
                    "the last line returns the change divided by eps: return change / eps.")
        if not _p3_is_number(answer):
            return ("%s gave %s, expected %s. %s." % (call, _p3_short(answer), _p3_dec(expected, 6), why),
                    "return one number, the slope: the change of the loss divided by eps.")
        hint = _p3_slip(fn, "slope_by_hop", _p3_hop_slips())
        if hint is None:
            hint = ("loss_before, then the hop original + eps, then loss_after, then the hop back, then "
                    "(loss_after - loss_before) / eps.")
        return ("%s gave %s, expected %s. %s." % (call, _p3_dec(answer, 6), _p3_dec(expected, 6), why), hint)
    return None


# Task B, slope

def _p3_slope_slips():
    P = _p3_inner
    return [
        (lambda n, q, r, p: 2 * n[p] * (P(n, q) - r),
         "you used the number itself, numbers[position]. The rule takes its partner, the number it meets in the "
         "inner product: the same position in the other list, partners[position]."),
        (lambda n, q, r, p: 2 * n[1 - p] * (P(n, q) - r),
         "you used the other number of the same list. The partner sits in the other list, at the same position: "
         "partners[position]."),
        (lambda n, q, r, p: 2 * q[1 - p] * (P(n, q) - r),
         "the partner sits in the other list at the same position, partners[position]. Your code took the other "
         "position."),
        (lambda n, q, r, p: q[p] * (P(n, q) - r),
         "your slopes are half as big: the 2 is missing. Return 2 * partner * difference."),
        (lambda n, q, r, p: q[p] * (r - P(n, q)),
         "the 2 is missing, and the sign is flipped. The difference is the prediction minus the rating, "
         "prediction - rating: return 2 * partner * difference."),
        (lambda n, q, r, p: 2 * q[p] * (r - P(n, q)),
         "the sign is flipped. The difference is the prediction minus the rating, prediction - rating, and the "
         "slope is 2 * partner * difference."),
        (lambda n, q, r, p: 2 * q[p] * P(n, q),
         "the rule takes the prediction minus the rating: difference = prediction - rating."),
        (lambda n, q, r, p: 2 * q[p] * (P(n, q) - _P3_RATING),
         "use rating, the number that the function gets, in place of %d. The check also tries another rating."
         % _P3_RATING),
    ]


def _p3_run_slope(fn):
    """None when slope passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P3_TESTS["slope"]:
        answer, error, copies = _p3_call(fn, inputs)
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p3_error_hint(error, "slope"))
        if not _p3_close_pair(copies[0], inputs[0]) or not _p3_close_pair(copies[1], inputs[1]):
            return ("%s changed the lists it got, to %s and %s" % (call, _p3_pair_text(copies[0]),
                                                                  _p3_pair_text(copies[1])),
                    "slope only measures: compute the prediction, the difference and the partner, and leave both "
                    "lists as they are.")
        if _p3_close(answer, expected):
            continue
        if answer is None:
            return ("%s gave no answer (None). Does the function end with a return line?" % call,
                    "the last line is return 2 * partner * difference.")
        if not _p3_is_number(answer):
            return ("%s gave %s, expected %s. %s." % (call, _p3_short(answer), _p3_dec(expected, 6), why),
                    "return one number: 2 times the partner times the difference.")
        hint = _p3_slip(fn, "slope", _p3_slope_slips())
        if hint is None and abs(answer - expected) <= 0.01 * max(1.0, abs(expected)):
            hint = ("close: a tiny hop gives nearly the slope. Task B asks for the exact slope, from the rule: "
                    "return 2 * partner * difference.")
        if hint is None:
            hint = ("difference = prediction - rating, partner = partners[position], then return 2 * partner * "
                    "difference.")
        return ("%s gave %s, expected %s. %s." % (call, _p3_dec(answer, 6), _p3_dec(expected, 6), why), hint)
    return None


# Task C, gradient_step

_P3_ONE_AT_A_TIME = ("close, but each slope must be measured on the guess as it was before the step. Measure all four "
                     "slopes first, in the first loop, and move the numbers in the second loop with the slopes you "
                     "measured: row[position] - alpha * row_slopes[position].")
_P3_STEP_SLIPS = [
    ("one at a time", _P3_ONE_AT_A_TIME),
    ("row first", _P3_ONE_AT_A_TIME),
    ("row, then column", _P3_ONE_AT_A_TIME),
    ("no swap",
     "each number of the column moved with the slope of the row's number at its position. For a number of the "
     "column, the column comes first: column_slopes[position] = slope(column, row, rating, position)."),
    ("plus",
     "the grasshopper went uphill: every number moved with its slope, and the loss grew. Move against the slope, "
     "with a minus: row[position] - alpha * row_slopes[position]."),
    ("no alpha",
     "every number lost its whole slope, a giant jump. The step is alpha times the slope: "
     "row[position] - alpha * row_slopes[position]."),
    ("divided",
     "the slope was divided by alpha, a giant jump. The step is alpha times the slope: "
     "row[position] - alpha * row_slopes[position]."),
]


def _p3_run_gradient_step(fn):
    """None when gradient_step passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P3_TESTS["gradient_step"]:
        row, column, rating, alpha = inputs
        expected_loss, expected_after = expected
        answer, error, copies = _p3_call(fn, inputs)
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p3_error_hint(error, "gradient_step"))
        after = [copies[0], copies[1]]
        wanted = "expected %s. %s." % (_p3_corner_text(expected_after), why)
        if not _p3_pair(after[0]) or not _p3_pair(after[1]):
            return ("%s left %s; %s" % (call, _p3_corner_text(after), wanted),
                    "move each number with row[position] = row[position] - alpha * row_slopes[position], and the "
                    "same for the column: the row and the column stay lists of two numbers.")
        if not _p3_close_corner(after, expected_after):
            got = "%s left %s; %s" % (call, _p3_corner_text(after), wanted)
            if _p3_close_corner(after, [row, column]):
                return (got, "no number moved. In the second loop, set row[position] = row[position] - alpha * "
                             "row_slopes[position], and the same for the column.")
            for how, hint in _P3_STEP_SLIPS:
                same = True
                for test in _P3_TESTS["gradient_step"]:
                    a, e, c = _p3_call(fn, test[0])
                    variant = _p3_step_variant(test[0][0], test[0][1], test[0][2], test[0][3], how)
                    if e is not None or not _p3_close_corner([c[0], c[1]], variant):
                        same = False
                        break
                if same:
                    return (got, hint)
            for whose in range(2):
                which = ["row", "column"][whose]
                moves = [not _p3_close(expected_after[whose][k], inputs[whose][k]) for k in range(2)]
                if _p3_close_pair(after[whose], inputs[whose]) and moves == [True, True]:
                    return (got, "the %s's two numbers did not move. The second loop moves every number: "
                                 "%s[position] = %s[position] - alpha * %s_slopes[position]."
                                 % (which, which, which, which))
            near = True
            for whose, position in _P3_ORDER:
                if abs(after[whose][position] - expected_after[whose][position]) > 0.05 * max(
                        1.0, abs(expected_after[whose][position])):
                    near = False
            if near:
                return (got, "close, but off. Measure every slope with slope, the rule of task B, all four before "
                             "any number moves, then move each number by alpha times its slope.")
            return (got, "first the four slopes, row_slopes[position] = slope(row, column, rating, position) and "
                         "column_slopes[position] = slope(column, row, rating, position); then row[position] = "
                         "row[position] - alpha * row_slopes[position], and the same for the column.")
        if answer is None:
            return ("%s moved the numbers right but gave no answer (None)" % call,
                    "end the function with return corner_loss(row, column, rating), the loss after the step.")
        if not _p3_close(answer, expected_loss):
            if _p3_close(answer, _p3_loss(row, column, rating)):
                return ("%s moved the numbers right but returned %s, the loss before the step; expected %s"
                        % (call, _p3_dec(answer, 6), _p3_dec(expected_loss, 6)),
                        "return the loss after the step: corner_loss(row, column, rating) on the last line, after "
                        "the numbers have moved.")
            return ("%s moved the numbers right but returned %s; expected the loss after the step, %s"
                    % (call, _p3_short(answer), _p3_dec(expected_loss, 6)),
                    "end the function with return corner_loss(row, column, rating), the loss after the step.")
    return None


# Task D, gradient_descent

def _p3_call_recorded(fn, inputs, calls):
    """Run gradient_descent with gradient_step replaced by a recorder, which makes the step right and notes the
    call (the rating and alpha it got). Gives back what _p3_call gives back."""
    space = getattr(fn, "__globals__", None)
    if not isinstance(space, dict):
        return _p3_call(fn, inputs)
    had = "gradient_step" in space
    saved = space.get("gradient_step")

    def recorder(row, column, rating, alpha):
        if len(calls) >= _P3_MAX_CALLS:
            raise _P3_Stop("many")
        if not _p3_pair(row) or not _p3_pair(column) or not _p3_is_number(rating) or not _p3_is_number(alpha):
            raise _P3_Stop("call")
        calls.append((rating, alpha))
        return _p3_step(row, column, rating, alpha)

    space["gradient_step"] = recorder
    try:
        return _p3_call(fn, inputs)
    finally:
        if had:
            space["gradient_step"] = saved
        else:
            del space["gradient_step"]


def _p3_run_gradient_descent(fn):
    """None when gradient_descent passes every test, else (what went wrong, hint)."""
    stop_hint = ("stop when the loss is below 0.0001: at the top of each round of the loop, if loss < 0.0001: "
                 "return losses. And loss must hold the loss after the latest step: loss = "
                 "gradient_step(row, column, rating, alpha).")
    for inputs, expected, call, why in _P3_TESTS["gradient_descent"]:
        row, column, rating, alpha, max_steps = inputs
        expected_losses, expected_end = expected
        calls = []
        answer, error, copies = _p3_call_recorded(fn, inputs, calls)
        after = [copies[0], copies[1]]
        if error is not None:
            if error == "STOP many":
                return ("%s made more than %d steps and was stopped. %s." % (call, _P3_MAX_CALLS, why),
                        "the loop makes at most max_steps steps, for step_number in range(max_steps), and stops "
                        "early when the loss is below 0.0001.")
            if error == "STOP call":
                return ("%s called gradient_step with something other than a row, a column, a rating and a step "
                        "size" % call,
                        "pass the four arguments in this order: gradient_step(row, column, rating, alpha).")
            return ("%s stopped with an error: %s" % (call, error), _p3_error_hint(error, "gradient_descent"))
        for used_rating, used_alpha in calls:
            if used_alpha != alpha:
                return ("%s called gradient_step with the step size %s; the function got %s"
                        % (call, _p3_short(used_alpha), _p3_short(alpha)),
                        "use alpha, the step size that gradient_descent gets: gradient_step(row, column, rating, "
                        "alpha). The check also tries other step sizes.")
            if used_rating != rating:
                return ("%s called gradient_step with the rating %s; the function got %s"
                        % (call, _p3_short(used_rating), _p3_short(rating)),
                        "use rating, the number that gradient_descent gets: gradient_step(row, column, rating, "
                        "alpha). The check also tries another rating.")
        if len(calls) > max_steps:
            return ("%s made %d steps, and max_steps was %d. %s." % (call, len(calls), max_steps, why),
                    "the given loop, for step_number in range(max_steps), makes at most max_steps steps: keep it "
                    "as it is.")
        if answer is None:
            return ("%s gave no answer (None). %s." % (call, why),
                    "return the list: return losses, both inside the stop and on the last line.")
        if not isinstance(answer, list):
            if _p3_is_number(answer):
                given = "the number %s" % _p3_dec(answer, 6)
            else:
                given = _p3_short(answer)
            return ("%s gave %s; expected the list of losses, %s. %s." % (call, given,
                                                                        _p3_list_text(expected_losses), why),
                    "return losses, the whole list, the loss at the start first.")
        for x in answer:
            if not _p3_is_number(x):
                return ("%s gave a list that holds %s; expected the losses, %s" % (call, _p3_short(x),
                                                                                  _p3_list_text(expected_losses)),
                        "each entry of losses is one number, a loss: losses.append(loss), with loss = "
                        "gradient_step(row, column, rating, alpha).")
        if _p3_close_list(answer, expected_losses):
            if not _p3_close_corner(after, expected_end):
                return ("%s gave the right losses, but left %s; expected %s"
                        % (call, _p3_corner_text(after), _p3_corner_text(expected_end)),
                        "gradient_descent changes the row and the column themselves, through gradient_step(row, "
                        "column, rating, alpha): work on row and column, the lists that the function gets.")
            continue
        got = "%s gave %s; expected %s. %s." % (call, _p3_list_text(answer), _p3_list_text(expected_losses), why)
        n, n_expected = len(answer), len(expected_losses)
        if not calls:
            return (got, "no step was made. In the loop, loss = gradient_step(row, column, rating, alpha) makes one "
                         "step and gives back the loss after it.")
        if n == n_expected - 1 and _p3_close_list(answer, expected_losses[1:]):
            return (got, "the loss at the start is missing: the list begins with it, losses = [loss], before the "
                         "loop.")
        if n == 1 and _p3_close(answer[0], expected_losses[0]):
            return (got, "losses holds only the loss at the start. Put each new loss at the end of the list, "
                         "losses.append(loss), right after the step.")
        repeats = True
        for x in answer:
            if not _p3_close(x, answer[0]):
                repeats = False
        if n > 1 and repeats and not _p3_close_list(answer, expected_losses[:n]):
            return (got, "the list repeats the loss at the start: loss = gradient_step(row, column, rating, alpha) "
                         "keeps the loss after each step, and losses.append(loss) puts it in the list.")
        if n > n_expected and _p3_close_list(answer[:n_expected], expected_losses):
            return ("%s went on after the loss was below 0.0001: the list holds %d losses, and %d were expected. "
                    "%s." % (call, n, n_expected, why), stop_hint)
        if n < n_expected and _p3_close_list(answer, expected_losses[:n]):
            return ("%s stopped after %s, with the loss at %s, above 0.0001. %s."
                    % (call, _p3_count(n - 1, "step"), _p3_loss_text(answer[-1]), why),
                    "return early only when the loss is below 0.0001: if loss < 0.0001: return losses. "
                    "Otherwise the loop goes on, and the last line, return losses, comes after max_steps steps.")
        return (got, "losses = [loss] before the loop; in each round, first the stop, then loss = "
                     "gradient_step(row, column, rating, alpha), then losses.append(loss).")
    return None


_P3_RUNNERS = {"slope_by_hop": _p3_run_slope_by_hop, "slope": _p3_run_slope,
               "gradient_step": _p3_run_gradient_step, "gradient_descent": _p3_run_gradient_descent}
_P3_NEEDS = {"slope_by_hop": [("corner_loss", "")],
             "slope": [("inner_product", "")],
             "gradient_step": [("slope", "task B"), ("corner_loss", "")],
             "gradient_descent": [("gradient_step", "task C"), ("corner_loss", "")]}
_P3_TASK = {"slope_by_hop": "A", "slope": "B", "gradient_step": "C", "gradient_descent": "D"}
_P3_GIVEN = {"inner_product": "in part 1 (or in this setup, when part 1 has not run)",
             "corner_loss": "at the top of part 3"}


def _p3_works(name):
    """True when the function called name exists and passes its tests."""
    fn = globals().get(name)
    if not callable(fn):
        return False
    if name in ("inner_product", "corner_loss"):
        if name == "inner_product":
            cases = [(([2, 1], [1, 2]), 4), (([-1, -2], [-1, 2]), -3), (([3], [-2]), -6), (([0.5, 1], [2, 0.5]), 1.5)]
        else:
            cases = []
            for row, column, rating in (([-3, -3], [-3, 2], 2), ([-1, -2], [-1, 2], 2), ([0.5, 1], [2, 0.5], 2),
                                        ([1, 1], [1, 1], -1)):
                cases.append(((row, column, rating), _p3_loss(row, column, rating)))
        for inputs, wanted in cases:
            answer, error, after = _p3_call(fn, inputs)
            if error is not None or not _p3_close(answer, wanted):
                return False
        return True
    if _p3_has_gap(fn):
        return False
    return _P3_RUNNERS[name](fn) is None


def _p3_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        if _p3_is_number(fn):
            got = "the number %s" % _p3_dec(fn, 6)
        else:
            got = _p3_short(fn)
        print("❌ check_%s needs the function %s itself, and it got %s. Write check_%s(%s), with no brackets "
              "after %s. If a cell set %s = ..., run the cell of task %s again to bring the function back."
              % (name, name, got, name, name, name, name, _P3_TASK[name]))
        return
    if _p3_has_gap(fn):
        print("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, run its "
              "cell, then run this check again." % name)
        return
    for other, place in _P3_NEEDS[name]:
        if not _p3_uses(fn, other) or _p3_works(other):
            continue
        if other in _P3_GIVEN:
            print("⏳ %s uses %s, which is missing or was changed. Run its cell %s again (it is given, so undo any "
                  "change to it), then run this check again." % (name, other, _P3_GIVEN[other]))
        else:
            print("⏳ %s uses your %s from %s, which does not work yet. Get the check of %s to ✅ first, then run "
                  "this check again." % (name, other, place, place))
        return
    failure = _P3_RUNNERS[name](fn)
    if failure is None:
        print("✅ %s works on %d tests. %s" % (name, len(_P3_TESTS[name]), _P3_SUCCESS[name]))
        return
    print("❌ " + failure[0])
    print("   💡 Hint: " + failure[1])


def check_slope_by_hop(fn):
    """Task A: tests slope_by_hop on the game's hop, two other hops, and other numbers, guesses and ratings."""
    try:
        _p3_check("slope_by_hop", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 3 again, then this cell." % error)


def check_slope(fn):
    """Task B: tests slope on the four numbers of two guesses, on another rating, on decimals and at a loss of 0."""
    try:
        _p3_check("slope", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 3 again, then this cell." % error)


def check_gradient_step(fn):
    """Task C: tests gradient_step on six guesses, with three step sizes: the row and the column after it, and
    the loss."""
    try:
        _p3_check("gradient_step", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 3 again, then this cell." % error)


def check_gradient_descent(fn):
    """Task D: tests gradient_descent on seven runs: the list of losses, the stop, max_steps, the row and the
    column at the end."""
    try:
        _p3_check("gradient_descent", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 3 again, then this cell." % error)


print("✅ Setup of part 3 complete.")

## The loss, given again

Part 3 measures the loss of a guess again and again. Here is part 2's `squared_loss`, as its solution writes it,
and `corner_loss`, the loss of a guess for the corner: the prediction, the inner product of Ida's row and
*Paddington*'s column, against her rating. Both are given so that this part works even if your part 2 is not finished.
Run the cell.

In [ ]:
# Given: part 2's squared_loss, as its solution writes it, and corner_loss, the loss of a guess for the
# corner, so that part 3 works on its own, even when part 2 is not finished
def squared_loss(prediction, rating):
    # prediction and rating are two numbers. Return their difference times itself.
    difference = prediction - rating
    loss = difference * difference
    return loss


def corner_loss(row, column, rating):
    # The loss of a guess for the corner: the prediction, the inner product of the row and the
    # column, against the rating. The two lists may come in either order: the inner product
    # multiplies them position by position, so the prediction stays the same.
    prediction = inner_product(row, column)
    loss = squared_loss(prediction, rating)
    return loss

## The stuck guess

The first half of the optional game starts from this guess: Ida -3 and -3, *Paddington* -3 and 2. The prediction is 3
and the rating 2, so the loss is 1. A hop of 1 on a number adds its partner to the prediction or takes it away,
and the partners here are 2 and 3 in size: the prediction jumps from 3 to 0, 1, 5 or 6, over the rating or away from it. No
whole hop lowers the loss: hops of 1 are stuck here.

In the code, as in the game, Ida's two numbers are her row, the list `ida_row`, and *Paddington*'s two are his column,
the list `paddington_column`, his first number on top. `show_corner` draws them the way the game does: the row on
the left, the column above, and the prediction where they meet.

In [ ]:
# The corner, Ida and Paddington, and the stuck guess, where no whole hop lowers the loss
rating = 2                                   # Ida gave Paddington 2: ratings[0][0] in part 2
ida_row = [-3, -3]                           # Ida's row: her first number, then her second
paddington_column = [-3, 2]                  # Paddington's column, from the top: his first number, then his second
show_corner(ida_row, paddington_column)
print("The prediction:", inner_product(ida_row, paddington_column))
print("The loss:      ", corner_loss(ida_row, paddington_column, rating))

## A valley between the hops

In the game the grasshopper stands on Ida's first number and watches only that number, with the other three held: the loss
against that one number is its **ground**. The cell below walks the ground at the two whole hops, one each way, and
at the stuck guess between them.

In [ ]:
# The ground of Ida's first number: the loss when only that number changes, the other three held
for x in [-4, -3, -2]:
    trial_row = [x, -3]                      # Ida's row with x as her first number
    print("Ida's first number", x, "  the loss", corner_loss(trial_row, paddington_column, rating))

**Reading the result.** The two whole hops land at the losses 16 and 4, both higher than 1. Between -3 and -2
the ground dips into a valley, and its bottom, the loss 0, lies at about -2.67. Hops of 1 jump over it. Smaller
hops can land in it.

## The grasshopper's hop

💭 **Think first.** The grasshopper stands on Ida's first number, -3, and hops a tiny step to the right, eps = 0.01. Will the
loss go up or down? Then run the cell below. It makes the hop by hand on a copy of Ida's row, `trial_row`, and
measures the loss before and after it.

In [ ]:
# The grasshopper's first hop, by hand: Ida's first number, from -3 to -3 + 0.01
trial_row = [-3, -3]                         # a copy of Ida's row at the stuck guess
loss_before = corner_loss(trial_row, paddington_column, rating)
trial_row[0] = -3 + 0.01                     # the hop
loss_after = corner_loss(trial_row, paddington_column, rating)
print("The loss before the hop:   ", loss_before)
print("The loss after the hop:    ", round(loss_after, 4))
print("The change:                ", round(loss_after - loss_before, 4))
print("The change per unit of hop:", round((loss_after - loss_before) / 0.01, 4))

**Reading the result.** The loss fell from 1 to 0.9409, so the ground goes down to the right. The change is 0.9409 - 1
= -0.0591 for a hop of 0.01, and per unit of hop that is -0.0591 / 0.01 = -5.91: the **slope** of the ground where the grasshopper
stands. A negative slope means downhill to the right.

`round(x, 4)` rounds a number to 4 decimals for printing. Python computes with binary fractions, so without it
the loss after the hop prints as 0.9409000000000013.

## 🎯 Task A. The slope from a tiny hop

Write `slope_by_hop(numbers, partners, rating, position, eps)`, the grasshopper's measurement on the number
`numbers[position]`, the way the cell above did it by hand: the loss before, the hop by `eps`, the loss after,
the hop back, and the change divided by `eps`. `numbers` is the list that holds the number, and `partners` is the
other list: for one of Ida's numbers, `ida_row` and then `paddington_column`; for one of *Paddington*'s, his column
first. `corner_loss` gives the same loss whichever list comes first, so one function serves all four numbers.
The grasshopper only feels the ground, so the function hops back and leaves both lists as it found them. For
Ida's first number, `slope_by_hop(ida_row, paddington_column, rating, 0, 0.01)` gives -5.91, with a little noise in the last
decimals.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# Four moves, as in the cell that hopped by hand:
# 1. the hop: the number becomes original + eps
# 2. the loss after the hop: corner_loss(numbers, partners, rating)
# 3. the hop back: the number becomes original again
# 4. the change, the loss after minus the loss before, divided by eps

In [ ]:
#@title 💡 Hint 2 for task A: the shape of the code (double-click to open) { display-mode: "form" }
#     numbers[position] = original + eps
#     loss_after = corner_loss(numbers, partners, rating)
#     numbers[position] = ...                      # the hop back, to the original
#     change = loss_after - loss_before
#     return ...                                   # the change per unit of hop

In [ ]:
# ✅ SOLUTION, task A: the slope from a tiny hop
def slope_by_hop(numbers, partners, rating, position, eps):
    # The grasshopper stands on the number numbers[position]. numbers is the list that holds it,
    # Ida's row or Paddington's column, and partners is the other list. It hops by eps, measures
    # how much the loss changed, and hops back. Return the change per unit of hop: the change
    # divided by eps.
    loss_before = corner_loss(numbers, partners, rating)   # the loss where the grasshopper stands
    original = numbers[position]                   # the number as it is now
    numbers[position] = original + eps             # the hop
    loss_after = corner_loss(numbers, partners, rating)    # the loss after the hop
    numbers[position] = original                   # the hop back: the list as it was
    change = loss_after - loss_before              # how much the loss changed
    return change / eps                            # the change per unit of hop

In [ ]:
check_slope_by_hop(slope_by_hop)

In [ ]:
# The grasshopper's hop of the game, with your function, on Ida's first number
eps = 0.01
print_hop(eps, slope_by_hop(ida_row, paddington_column, rating, 0, eps))

**Reading the result.** The function measures -5.91, the slope the cell above found by hand. It is negative, so
the ground goes down to the right, and its size, 5.91, says how steep the ground is there.

## 🎯 Task B. The slope from a formula

In the game, the grasshopper's hop of 0.01 on Ida's first number changed only one product of the inner product, the
number times its partner, Paddington's first number, -3: from 9 to 8.97. The prediction became 2.97,
the difference (the prediction minus the rating) 0.97, and the loss 0.9409. The loss changed by
-0.0591, and -0.0591 divided by the hop, 0.01, is the slope -5.91. A rule gives the slope without a
hop: 2 times the partner times the difference, 2 times -3 times 1, which is -6. The
hop of 0.01 measured -5.91; the rule gives the slope right at -3.

The same holds for every guessed number: **the slope is 2 times its partner times the prediction minus the
rating.** The partner of a number is the number it meets in the inner product, at the same position in the other
list: Ida's first number meets *Paddington*'s first, her second meets his second. In the code it is `partners[position]`.

Write `slope(numbers, partners, rating, position)`, with the two lists in the order of task A. For Ida's first number at the
stuck guess, `slope(ida_row, paddington_column, rating, 0)` gives -6.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The difference is the prediction minus the rating: 3 - 2 = 1 at the stuck guess.
# The partner of numbers[position] sits in the other list, partners, at the same position.
# For Ida's first number, ida_row[0], the partner is paddington_column[0], Paddington's first number.

In [ ]:
#@title 💡 Hint 2 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#     difference = prediction - rating
#     partner = partners[position]
#     return 2 * ... * ...                        # 2 times the partner times the difference

In [ ]:
# ✅ SOLUTION, task B: the slope from a formula
def slope(numbers, partners, rating, position):
    # The slope of the loss at the number numbers[position]: 2 times its partner times the
    # prediction minus the rating. Its partner is the number it meets in the inner product, at
    # the same position in the other list: partners[position].
    prediction = inner_product(numbers, partners)
    difference = prediction - rating               # the prediction minus the rating
    partner = partners[position]                   # the number it meets in the inner product
    return 2 * partner * difference

In [ ]:
check_slope(slope)

In [ ]:
# The four slopes at the stuck guess: from the formula, and from the game's hop of 0.01
for position in range(2):                    # Ida's two numbers: her row first
    from_formula = slope(ida_row, paddington_column, rating, position)
    from_hop = slope_by_hop(ida_row, paddington_column, rating, position, 0.01)
    print_slopes("Ida", position, from_formula, from_hop)
for position in range(2):                    # Paddington's two numbers: his column first
    from_formula = slope(paddington_column, ida_row, rating, position)
    from_hop = slope_by_hop(paddington_column, ida_row, rating, position, 0.01)
    print_slopes("Paddington", position, from_formula, from_hop)

**Reading the result.** The formula gives -6, 4, -6 and -6, and the hop of 0.01 agrees to within 0.1. Three slopes are negative:
those numbers should grow. Ida's second number has the slope 4: its partner, Paddington's second number, is 2, so a hop up makes the loss bigger, and
this number should shrink. A slope says which way is downhill, and how steep it is.

## 🎯 Task C. One step downhill

The grasshopper jumps against the slope: **the new number is the old number minus alpha times its slope.** With
a negative slope the ground goes down to the right, so the number grows; with a positive slope it shrinks. alpha
is the size of the step, 0.02 in the game: small, because a slope can be large, and a step too far lands on the
other side of the valley. For Ida's first number: `-3 - 0.02 * (-6)` is -2.88.

All four numbers move in one step, and every slope is measured on the same guess, before any number moves. So
the function has two loops: the first measures all four slopes, the second moves all four numbers. Each loop goes
through the two positions, and at each position it handles one number of the row and one of the column. The four
slopes together are called the **gradient**: at the stuck guess [-6, 4] for Ida's row and [-6, -6] for *Paddington*'s column.

Write `gradient_step(row, column, rating, alpha)`. It changes `row` and `column` themselves and returns the loss
after the step. From the stuck guess with alpha 0.02: Ida -2.88 and -3.08, Paddington -2.88 and 2.12, and the loss 0.0553.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# The first loop only measures. For a number of the row: slope(row, column, rating, position);
# for a number of the column the column comes first: slope(column, row, rating, position).
# No number moves in it, so all four slopes belong to the same guess.
# The second loop moves: each number takes alpha times its slope away from itself.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
# For Ida's first number at the stuck guess: -3 - 0.02 * (-6) = -2.88.
# In Python, for a number of the row: row[position] - alpha * row_slopes[position]

In [ ]:
#@title 💡 Hint 3 for task C: the shape of the code (double-click to open) { display-mode: "form" }
#         row_slopes[position] = slope(row, column, rating, position)
#         column_slopes[position] = slope(column, row, rating, position)
#     ...
#         row[position] = row[position] - ... * ...
#         column[position] = ...

In [ ]:
# ✅ SOLUTION, task C: one step downhill for all four numbers
def gradient_step(row, column, rating, alpha):
    # One step downhill for all four numbers, the row's two and the column's two. First measure the
    # four slopes on the guess as it is now; then move every number against its slope: the old
    # number minus alpha times its slope. Return the loss after the step.
    row_slopes = [0, 0]                            # room for the slopes of the row's two numbers
    column_slopes = [0, 0]                         # and for the slopes of the column's two numbers
    # First: all four slopes, before any number moves
    for position in range(2):
        row_slopes[position] = slope(row, column, rating, position)
        column_slopes[position] = slope(column, row, rating, position)
    # Then: every number moves against its slope
    for position in range(2):
        row[position] = row[position] - alpha * row_slopes[position]
        column[position] = column[position] - alpha * column_slopes[position]
    return corner_loss(row, column, rating)        # the loss after the step

In [ ]:
check_gradient_step(gradient_step)

In [ ]:
# One step downhill from the stuck guess, with the step size of the game
alpha = 0.02
ida_row = [-3, -3]                           # the stuck guess again
paddington_column = [-3, 2]
loss_after = gradient_step(ida_row, paddington_column, rating, alpha)
print_step(1, ida_row, paddington_column, loss_after)
show_corner(ida_row, paddington_column)

**Reading the result.** Every number moved a little against its slope: Ida -2.88 and -3.08, Paddington -2.88 and 2.12. The loss fell from 1 to 0.0553 in one
step, from a guess where whole hops could not move at all. The prediction is now 1.7648, a little below the rating
2.

## 🎯 Task D. Again and again

Gradient descent repeats the step: measure the slopes, step downhill, and go again. The loss shrinks step after
step, and below 0.0001 it counts as 0. Write `gradient_descent(row, column, rating, alpha, max_steps)`. It makes at
most `max_steps` steps, with `for step_number in range(max_steps)`, and stops early, with `return`, once the loss
is below 0.0001. It changes `row` and `column` themselves and returns the list of losses, the loss at the start
first, so that you can watch it fall. A given line, `print_step`, prints each step: the loss, the prediction and
the four numbers.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# losses starts as a list that holds one number, the loss at the start: losses = [loss].
# Each round of the loop does three things: first the stop, then one step, then the new loss
# goes at the end of the list.

In [ ]:
#@title 💡 Hint 2 for task D (double-click to open) { display-mode: "form" }
# The stop: if loss < 0.0001: return losses. It comes first in the round, so a guess that is
# already good makes no step at all.
# One step: loss = gradient_step(row, column, rating, alpha) moves the four numbers and gives
# back the loss after the step.

In [ ]:
#@title 💡 Hint 3 for task D: the shape of the code (double-click to open) { display-mode: "form" }
#     losses = [loss]
#     for step_number in range(max_steps):
#         if loss < 0.0001:
#             return losses
#         loss = gradient_step(row, column, rating, alpha)
#         losses.append(...)
#         print_step(step_number + 1, row, column, loss)
#     return losses

In [ ]:
# ✅ SOLUTION, task D: gradient descent on the corner
def gradient_descent(row, column, rating, alpha, max_steps):
    # Step downhill again and again. Stop when the loss is below 0.0001, where it counts as 0, or
    # after max_steps steps. Return the list of losses, the loss at the start first.
    loss = corner_loss(row, column, rating)        # the loss at the start
    losses = [loss]                                # the list of losses, the start first
    for step_number in range(max_steps):
        # Small enough: the loss counts as 0, so stop
        if loss < 0.0001:
            return losses
        loss = gradient_step(row, column, rating, alpha)   # one step downhill, and the loss after it
        losses.append(loss)
        print_step(step_number + 1, row, column, loss)
    return losses

In [ ]:
check_gradient_descent(gradient_descent)

In [ ]:
# Gradient descent from the stuck guess, at most 100 steps
ida_row = [-3, -3]                           # the stuck guess again
paddington_column = [-3, 2]
losses = gradient_descent(ida_row, paddington_column, rating, alpha, 100)
show_losses(losses)
show_corner(ida_row, paddington_column)

**Reading the result.** 4 steps, and the loss goes 1, 0.0553, 0.0028, 0.00014, then about 0. The prediction swings around the rating, 1.76, 2.05, 1.99 and 2.00, and settles on
2. Whole hops could not move from this guess: they jumped over the valley. The small steps, guided by the
slopes, land in it.

## Five experiments

💭 **Think first** before you run each cell below: what will the loss do? Each cell writes its guess into
`ida_row` and `paddington_column`.

1. The first guess of part 2, Ida -1 and -2 with *Paddington* -1 and 2, the loss 25. How many steps will gradient
   descent need?
2. Your own guess from the game.
3. All four numbers at 0, the loss 4. What slopes will the grasshoppers feel?
4. A step that is too big: alpha 0.1 in place of 0.02, from the stuck guess.
5. A flipped guess: Paddington's numbers are Ida's with the signs flipped, Ida 1 and 2 with *Paddington* -1 and -2.

In [ ]:
# Experiment 1: part 2's first guess
ida_row = [-1, -2]
paddington_column = [-1, 2]
losses = gradient_descent(ida_row, paddington_column, rating, alpha, 100)
show_losses(losses)

**Reading the result.** 16 steps, from 25 to about 0. From the stuck guess, with the loss 1, it took 4 steps.

**Experiment 2, your own guess.** At the end of part two of the game you typed four numbers of your own for Ida
and *Paddington*. Type them into the cell below, in place of the Matchmaker's first guess, and run the cell. If you
skipped that step of the game, pick any four numbers from -2 to 2. Run the cell again whenever you change the numbers.

### 🎯 Your turn

In [ ]:
# ✅ AN EXAMPLE: the slowest guess with numbers from -2 to 2, 42 steps (yours may differ)
ida_row = [-1, 0]                            # Ida's first and second number
paddington_column = [0, 0]                   # Paddington's first and second number
losses = gradient_descent(ida_row, paddington_column, rating, alpha, 50)
show_losses(losses)
show_corner(ida_row, paddington_column)

**Reading the result.** The loss never rises from one step to the next. From every guess of numbers from -2 to
2 it gets below 0.0001 within 42 steps, except from 25 guesses: the flipped guesses of experiment 5, all zeros among
them. Where your numbers end depends on where they start: many pairs of a row and a column give the inner
product 2.

In [ ]:
# Experiment 3: all four numbers at 0, 5 steps
ida_row = [0, 0]
paddington_column = [0, 0]
losses = gradient_descent(ida_row, paddington_column, rating, alpha, 5)
show_losses(losses)

**Reading the result.** The loss stays 4. At 0 every partner is 0, so every slope, 2 times the partner times
the prediction minus the rating, is 0: the ground is flat there, and the grasshoppers feel no slope to follow.
No number moves.

In [ ]:
# Experiment 4: a step that is too big, alpha = 0.1, 3 steps from the stuck guess
ida_row = [-3, -3]
paddington_column = [-3, 2]
losses = gradient_descent(ida_row, paddington_column, rating, 0.1, 3)
show_losses(losses)

**Reading the result.** The losses grow: 1, 25.8064, 486.5594, 77270.0032. With alpha 0.1 the first step takes the prediction from 3 to -3.08, far
past the rating 2, and the loss lands at 25.8. There the slopes are steeper, so the next jump is longer still. A
small alpha, 0.02, makes small jumps, and the loss falls at every step.

In [ ]:
# Experiment 5: Paddington's numbers are Ida's with the signs flipped, 30 steps
ida_row = [1, 2]
paddington_column = [-1, -2]
losses = gradient_descent(ida_row, paddington_column, rating, alpha, 30)
show_losses(losses)
show_corner(ida_row, paddington_column)

**Reading the result.** The loss falls at every step, from 49 down towards 4, and stays above 0.0001: the numbers
of the row and the column slide towards 0. 💭 Why does it stop near 4? Think first, then open the folded answer
below.

In [ ]:
#@title 💭 Answer: why the flipped guess stops near the loss 4 (double-click to open) { display-mode: "form" }
# Paddington's column is Ida's row with every sign flipped, so both products of the inner product are
# negative, and the prediction is below 0, far from the rating 2.
# Every step shrinks the row and the column by the same factor, and they stay flipped, so the
# prediction can never climb to 2. The best a flipped pair can do is all zeros, where the
# prediction is 0 and the loss is 4. There every slope is 0: flat ground, as in experiment 3.

## Back to the game

That is part 3: your `gradient_descent` feels the slope of every number and steps downhill, a small step at a
time, and gets out of the stuck guess. The second half of the optional game, "Many customers, many films", goes
from the corner to the whole table with the same method, one grasshopper for every guessed number. Part 4 of
this notebook comes after it.

# Part 4. Many customers, many films

In the second half of the optional game the Matchmaker went from one corner to the whole table, with the same
method: first three customers and one film, the column of *Paddington*, then both films. The loss of a table is the **average** of the
losses of its ratings. A guessed number sits in several predictions, so every rating it sits in has a say in its
slope. And every number moves at once, again and again, until the product of the two guessed matrices gives the
ratings.

In this part you teach the computer to do that:

- **Task A**: the average loss of a table;
- **Task B**: the slope of one customer's number;
- **Task C**: the slope of one film's number;
- **Task D**: one step for every number at once;
- **Task E**: `factorize`, step after step, until the product gives the ratings.

Then the loop runs on part 1's scores, and three experiments. As in the game, the films' matrix has one row per
number and one column per film, so film `j`'s numbers are its column. Part 4 stands on its own: it uses part 1's
functions, given again below, and nothing from parts 2 and 3. Run its setup cell first.

In [ ]:
#@title Setup of part 4 (run this first) { display-mode: "form" }
# The plumbing of part 4: show_guess, which prints every rating of a table with its prediction and its loss,
# print_loss, show_slopes, print_progress and show_run, which print results in words, and the checks of tasks
# A to E. You do not need to read it. A check prints one short message and never stops the notebook.

# Part 4 uses show_matrix from part 1's setup, which is in the main notebook. This cell defines a plain one when
# it is missing, so that part 4 works on its own. Part 1's functions come again in the cell after this one.
if "show_matrix" not in globals() or "plain stand-in" in str(getattr(show_matrix, "__doc__", "")):

    def show_matrix(matrix, row_names=None, column_names=None):
        """A plain stand-in for the show_matrix of part 1: the table as text, decimals rounded to 2 places."""
        try:
            fine = isinstance(matrix, list) and len(matrix) > 0
            if fine:
                for row in matrix:
                    if not isinstance(row, list) or len(row) == 0 or len(row) != len(matrix[0]):
                        fine = False
                    else:
                        for x in row:
                            if x is Ellipsis or x is None or isinstance(x, (list, tuple)):
                                fine = False
            if not fine:
                print("⏳ Nothing to show yet: %s. Finish the 🎯 cell above, run it, then run this cell again."
                      % _p4_short(matrix))
                return
            rows, columns = len(matrix), len(matrix[0])
            if not isinstance(row_names, (list, tuple)) or len(row_names) != rows:
                row_names = [str(i) for i in range(rows)]
            if not isinstance(column_names, (list, tuple)) or len(column_names) != columns:
                column_names = [str(j) for j in range(columns)]
            entries = []
            for i in range(rows):
                line = []
                for j in range(columns):
                    if isinstance(matrix[i][j], float):
                        line.append("%.2f" % matrix[i][j])
                    else:
                        line.append(str(matrix[i][j]))
                entries.append(line)
            width = max(len(str(name)) for name in row_names)
            widths = []
            for j in range(columns):
                w = len(str(column_names[j]))
                for i in range(rows):
                    w = max(w, len(entries[i][j]))
                widths.append(w)
            print(" " * width + "".join("   " + str(column_names[j]).rjust(widths[j]) for j in range(columns)))
            for i in range(rows):
                print(str(row_names[i]).ljust(width)
                      + "".join("   " + entries[i][j].rjust(widths[j]) for j in range(columns)))
        except Exception as error:
            print("The table could not be shown (%s)." % error)

_P4_TOL = 0.0001                 # below this the average loss counts as 0
_P4_MARGIN = 1e-9                 # two decimal numbers count as equal within this relative margin
_P4_MAX_CALLS = 5000              # the check of task E stops a loop that makes more steps
_P4_POSITION_NAMES = ["first", "second", "third", "fourth", "fifth", "sixth"]


def _p4_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 80:
        text = text[:77] + "..."
    return text


def _p4_is_number(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


def _p4_dec(x, places):
    """x rounded to at most places decimals, without trailing zeros: -6.6667, 0.0553, 4."""
    if not _p4_is_number(x):
        return _p4_short(x)
    if x != x or x == float("inf") or x == float("-inf"):
        return str(x)
    text = "%.*f" % (places, x)
    if "." in text:
        text = text.rstrip("0").rstrip(".")
    if text == "-0":
        text = "0"
    return text


def _p4_loss_text(x):
    """A loss as text: at most 4 decimals, 5 for a small one, 1 for a huge one, and about 0 below 0.0001."""
    if _p4_is_number(x) and 0 <= x < _P4_TOL:
        return "about 0"
    if _p4_is_number(x) and 0 < x < 0.001:
        return _p4_dec(x, 5)
    if _p4_is_number(x) and x >= 1000:
        return _p4_dec(x, 1)
    return _p4_dec(x, 4)


def _p4_count(n, word):
    """1 step, 52 steps"""
    if n == 1:
        return "1 " + word
    return "%d %ss" % (n, word)


def _p4_waiting(value):
    """Why a value is not a number yet, in words."""
    if value is Ellipsis:
        return "it is still ..., a gap in a 🎯 cell"
    if value is None:
        return "it is None (does the function end with a return line?)"
    return "it is %s" % _p4_short(value)


def _p4_matrix(value):
    """True when value is a list of rows of numbers, every row as long as the first."""
    if not isinstance(value, list) or len(value) == 0:
        return False
    for row in value:
        if not isinstance(row, list) or len(row) == 0 or len(row) != len(value[0]):
            return False
        for x in row:
            if not _p4_is_number(x):
                return False
    return True


def _p4_copy(matrix):
    out = []
    for row in matrix:
        out.append(list(row))
    return out


def _p4_names(names, n, word):
    """The names of n rows: the names when they fit, else customer 0, customer 1, ..."""
    if isinstance(names, (list, tuple)) and len(names) == n:
        return [str(x) for x in names]
    return ["%s %d" % (word, k) for k in range(n)]


# The arithmetic, kept here for the helpers and the checks. films has one row per number and one column per film:
# film j's numbers are its column, and its number at a position is films[position][j].

def _p4_inner(row_a, row_b):
    total = 0
    for i in range(len(row_a)):
        total = total + row_a[i] * row_b[i]
    return total


def _p4_column(matrix, j):
    numbers = []
    for row in matrix:
        numbers.append(row[j])
    return numbers


def _p4_loss(customers, films, ratings):
    """The average loss of a table: the prediction minus the rating, squared, averaged over every rating."""
    total = 0
    for i in range(len(customers)):
        for j in range(len(films[0])):
            difference = _p4_inner(customers[i], _p4_column(films, j)) - ratings[i][j]
            total = total + difference * difference
    return total / (len(customers) * len(films[0]))


def _p4_customer_slope(customers, films, ratings, i, position):
    total = 0
    for j in range(len(films[0])):
        difference = _p4_inner(customers[i], _p4_column(films, j)) - ratings[i][j]
        total = total + 2 * films[position][j] * difference
    return total / (len(customers) * len(films[0]))


def _p4_film_slope(customers, films, ratings, j, position):
    total = 0
    for i in range(len(customers)):
        difference = _p4_inner(customers[i], _p4_column(films, j)) - ratings[i][j]
        total = total + 2 * customers[i][position] * difference
    return total / (len(customers) * len(films[0]))


def _p4_all_slopes(customers, films, ratings):
    """The slope of every number, each in the place of its number: customer_slopes in the shape of customers,
    film_slopes in the shape of films."""
    customer_slopes = []
    for i in range(len(customers)):
        row = []
        for position in range(len(customers[i])):
            row.append(_p4_customer_slope(customers, films, ratings, i, position))
        customer_slopes.append(row)
    film_slopes = []
    for position in range(len(films)):
        row = []
        for j in range(len(films[position])):
            row.append(_p4_film_slope(customers, films, ratings, j, position))
        film_slopes.append(row)
    return customer_slopes, film_slopes


def _p4_step(customers, films, ratings, alpha):
    """One step of every number, in place, as table_step should do it. Gives back the average loss after it."""
    customer_slopes, film_slopes = _p4_all_slopes(customers, films, ratings)
    for i in range(len(customers)):
        for position in range(len(customers[i])):
            customers[i][position] = customers[i][position] - alpha * customer_slopes[i][position]
    for position in range(len(films)):
        for j in range(len(films[position])):
            films[position][j] = films[position][j] - alpha * film_slopes[position][j]
    return _p4_loss(customers, films, ratings)


def _p4_step_variant(customers, films, ratings, alpha, how):
    """The two matrices after one step done with a slip, as copies: how is "one at a time", "customers first",
    "plus", "no alpha", "divided", "only customers" or "only films"."""
    c = _p4_copy(customers)
    f = _p4_copy(films)
    if how == "one at a time":
        for i in range(len(c)):
            for position in range(len(c[i])):
                c[i][position] = c[i][position] - alpha * _p4_customer_slope(c, f, ratings, i, position)
        for position in range(len(f)):
            for j in range(len(f[position])):
                f[position][j] = f[position][j] - alpha * _p4_film_slope(c, f, ratings, j, position)
        return c, f
    if how == "customers first":
        customer_slopes = _p4_all_slopes(c, f, ratings)[0]
        for i in range(len(c)):
            for position in range(len(c[i])):
                c[i][position] = c[i][position] - alpha * customer_slopes[i][position]
        film_slopes = _p4_all_slopes(c, f, ratings)[1]
        for position in range(len(f)):
            for j in range(len(f[position])):
                f[position][j] = f[position][j] - alpha * film_slopes[position][j]
        return c, f
    customer_slopes, film_slopes = _p4_all_slopes(c, f, ratings)
    factor = {"plus": -alpha, "no alpha": 1, "divided": 1 / alpha, "only customers": alpha,
              "only films": alpha}[how]
    if how != "only films":
        for i in range(len(c)):
            for position in range(len(c[i])):
                c[i][position] = c[i][position] - factor * customer_slopes[i][position]
    if how != "only customers":
        for position in range(len(f)):
            for j in range(len(f[position])):
                f[position][j] = f[position][j] - factor * film_slopes[position][j]
    return c, f


# The helpers that print results in words

def show_guess(customers, films, ratings, customer_names=None, film_names=None):
    """Every rating of a table, one line each, with its prediction (the inner product of the customer's row and
    the film's column) and its loss (the prediction minus the rating, squared), at most 2 decimals. films has one
    row per number and one column per film."""
    try:
        if not _p4_matrix(customers) or not _p4_matrix(films) or not _p4_matrix(ratings):
            print("⏳ Nothing to show yet: the guess or the ratings are not tables of numbers yet.")
            return
        if len(ratings) != len(customers) or len(ratings[0]) != len(films[0]) or len(customers[0]) != len(films):
            print("⏳ These do not fit together: customers with %s of %s, films with %s and %s (one row per number, "
                  "one column per film), and ratings with %s and %s."
                  % (_p4_count(len(customers), "row"), _p4_count(len(customers[0]), "number"),
                     _p4_count(len(films), "row"), _p4_count(len(films[0]), "column"),
                     _p4_count(len(ratings), "row"), _p4_count(len(ratings[0]), "column")))
            return
        c_names = _p4_names(customer_names, len(customers), "customer")
        f_names = _p4_names(film_names, len(films[0]), "film")
        lines = []
        for i in range(len(customers)):
            for j in range(len(films[0])):
                prediction = _p4_inner(customers[i], _p4_column(films, j))
                difference = prediction - ratings[i][j]
                lines.append(["%s and %s" % (c_names[i], f_names[j]), _p4_dec(prediction, 2),
                              _p4_dec(ratings[i][j], 2), _p4_dec(difference * difference, 2)])
        widths = []
        for k in range(3):
            w = 0
            for line in lines:
                w = max(w, len(line[k]))
            widths.append(w)
        for line in lines:
            print("%s   the prediction %s   the rating %s   the loss %s" % (
                line[0].ljust(widths[0]), line[1].rjust(widths[1]), line[2].rjust(widths[2]), line[3]))
    except Exception as error:
        print("The guess could not be shown (%s)." % error)


def print_loss(text, loss):
    """One line: the text and an average loss, at most 4 decimals."""
    try:
        if not _p4_is_number(loss):
            print("%s: ⏳ no loss yet, %s. The checks of the 🎯 tasks above say more." % (text, _p4_waiting(loss)))
            return
        print("%s: %s" % (text, _p4_loss_text(loss)))
    except Exception:
        pass


def show_slopes(customers, films, ratings, customer_names=None, film_names=None):
    """The slope of every number of both matrices, from your customer_slope (task B) and film_slope (task C),
    as two tables in the places of the numbers: the customers' slopes one row per customer, the films' slopes one
    column per film, as the films' numbers stand."""
    try:
        if not _p4_matrix(customers) or not _p4_matrix(films) or not _p4_matrix(ratings):
            print("⏳ Nothing to show yet: the guess or the ratings are not tables of numbers yet.")
            return
        # The places of the numbers, (index, position) in the shape of each matrix: customer i's number at a
        # position, and film j's number at a position, which stands in row position and column j of films
        places = {"customer_slope": [[(i, position) for position in range(len(customers[i]))]
                                     for i in range(len(customers))],
                  "film_slope": [[(j, position) for j in range(len(films[position]))]
                                 for position in range(len(films))]}
        tables = []
        for name, task in (("customer_slope", "B"), ("film_slope", "C")):
            fn = globals().get(name)
            if not callable(fn):
                print("⏳ %s is not there yet: run the cell of task %s, then this cell again." % (name, task))
                return
            slopes = []
            for row_of_places in places[name]:
                row = []
                for index, position in row_of_places:
                    answer, error, after = _p4_call(fn, (customers, films, ratings, index, position))
                    if error is not None or not _p4_is_number(answer):
                        if error is not None:
                            why = "it stopped with an error"
                        else:
                            why = _p4_waiting(answer)
                        print("⏳ No slopes yet from %s: %s. Finish task %s, run its cell, then run this cell "
                              "again." % (name, why, task))
                        return
                    row.append(answer)
                slopes.append(row)
            tables.append(slopes)
        numbers = []
        for position in range(len(customers[0])):
            numbers.append(_P4_POSITION_NAMES[position] + " number")
        print("The slopes of the customers' numbers:")
        show_matrix(tables[0], _p4_names(customer_names, len(customers), "customer"), numbers)
        print("The slopes of the films' numbers:")
        show_matrix(tables[1], numbers, _p4_names(film_names, len(films[0]), "film"))
    except Exception as error:
        print("The slopes could not be shown (%s)." % error)


def print_progress(step_number, loss):
    """One line for a step of factorize: the step and the average loss after it. It prints steps 1, 2 and 3,
    then every 10th step up to 100, then every 100th, so that a long run stays short to read. It prints nothing
    while the loss is still ..., a gap in the 🎯 cell of task E, and one calm line at step 1 when the loss is
    something else."""
    try:
        if not _p4_is_number(loss):
            if loss is not Ellipsis and step_number == 1:
                print("Step   1:  ⏳ no loss yet: %s. The checks of tasks D and E say more." % _p4_waiting(loss))
            return
        if step_number <= 3 or (step_number <= 100 and step_number % 10 == 0) or step_number % 100 == 0:
            print("Step %3s:  the average loss %s" % (step_number, _p4_loss_text(loss)))
    except Exception:
        pass


def show_run(losses):
    """The list of losses that factorize gives back, in one line: the number of steps, then the average losses,
    the loss at the start first (all of them, or the first five and the last three)."""
    try:
        if losses is Ellipsis:
            print("⏳ Nothing to show yet: losses is still ..., a gap in a 🎯 cell. Finish task E and any 🎯 line "
                  "of this cell, then run this cell again.")
            return
        if losses is None:
            print("⏳ Nothing to show yet: factorize gave back None. Does it end with return losses? The check of "
                  "task E says more.")
            return
        if not isinstance(losses, list) or len(losses) == 0:
            if isinstance(losses, list):
                reason = "the list of losses is empty"
            else:
                reason = "losses is %s, and a list of losses was expected" % _p4_short(losses)
            print("⏳ Nothing to show yet: %s. The check of task E says more." % reason)
            return
        for x in losses:
            if not _p4_is_number(x):
                print("⏳ The list of losses still holds %s. The check of task E says more." % _p4_short(x))
                return
        texts = []
        for x in losses:
            texts.append(_p4_loss_text(x))
        if len(texts) > 9:
            texts = texts[:5] + ["..."] + texts[-3:]
        steps = len(losses) - 1
        if steps == 0:
            print("No step. The average loss: %s" % texts[0])
        else:
            print("%s. The average losses: %s" % (_p4_count(steps, "step"), ", ".join(texts)))
    except Exception as error:
        print("The losses could not be shown (%s)." % error)


# The checks

# Each test: the inputs, the expected answer, the call as you would type it, and why that is the answer.
_P4_TESTS = {
    "average_loss": [
        (([[-1, -2], [-2, 0], [2, 2]], [[-1], [2]], [[2], [-1], [1]]), 11.666666666666666,
         'average_loss(customers, column_films, column_ratings)',
         'The column, from the first guess: the losses 25, 9 and 1, and their average 35 / 3, about 11.6667'),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]]), 10.666666666666666,
         'average_loss(customers, films, ratings)',
         'The whole table, from the first guess: the six losses 25, 9, 9, 4, 1 and 16 add up to 64, and 64 / 6 is about 10.6667'),
        (([[-2, 0, 2], [2, -2, -1], [2, 2, 2]], [[1, 1, -1, -2], [2, 2, -2, -2], [2, 1, 2, 0]], [[7, 0, -7, -6], [-4, -6, 10, 0], [-1, 7, -2, 2]]), 52.083333333333336,
         'average_loss(score_customers, score_films, scores)',
         "Part 1's scores with a first guess of three numbers each: 12 ratings, and the average of their losses is about 52.0833"),
        (([[-1, -2]], [[-1], [2]], [[2]]), 25.0,
         'average_loss([[-1, -2]], [[-1], [2]], [[2]])',
         'A table of one rating, Ida and Paddington: the average of one loss is that loss, 25'),
        (([[2, 1], [-1, 2], [1, -2]], [[1, 0], [0, 1]], [[2, 1], [-1, 2], [1, -2]]), 0.0,
         'average_loss(ratings, [[1, 0], [0, 1]], ratings)',
         'Every prediction gives its rating exactly, so every loss is 0, and so is their average'),
        (([[-0.9333333333333333, -1.1333333333333333], [-1.5333333333333334, -0.2666666666666667], [1.5333333333333334, 1.6]], [[-1.0666666666666667, -1.0], [1.2, -0.06666666666666676]], [[2, 1], [-1, 2], [1, -2]]), 1.9659325102880663,
         'average_loss(customers, films, ratings), one step after the first guess',
         'A guess with decimals: the average is about 1.9659'),
    ],
    "customer_slope": [
        (([[-1, -2], [-2, 0], [2, 2]], [[-1], [2]], [[2], [-1], [1]], 0, 0), 3.3333333333333335,
         'customer_slope(customers, column_films, column_ratings, 0, 0)',
         "Ida's first number on the column: the one rating it sits in gives it 10, the other 2 give 0, and 10 / 3 is 3.3333"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1], [2]], [[2], [-1], [1]], 0, 1), -6.666666666666667,
         'customer_slope(customers, column_films, column_ratings, 0, 1)',
         "Ida's second number on the column: the one rating it sits in gives it -20, the other 2 give 0, and -20 / 3 is -6.6667"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1], [2]], [[2], [-1], [1]], 1, 0), -2.0,
         'customer_slope(customers, column_films, column_ratings, 1, 0)',
         "Raj's first number on the column: the one rating it sits in gives it -6, the other 2 give 0, and -6 / 3 is -2"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1], [2]], [[2], [-1], [1]], 1, 1), 4.0,
         'customer_slope(customers, column_films, column_ratings, 1, 1)',
         "Raj's second number on the column: the one rating it sits in gives it 12, the other 2 give 0, and 12 / 3 is 4"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1], [2]], [[2], [-1], [1]], 2, 0), -0.6666666666666666,
         'customer_slope(customers, column_films, column_ratings, 2, 0)',
         "Elsa's first number on the column: the one rating it sits in gives it -2, the other 2 give 0, and -2 / 3 is -0.6667"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1], [2]], [[2], [-1], [1]], 2, 1), 1.3333333333333333,
         'customer_slope(customers, column_films, column_ratings, 2, 1)',
         "Elsa's second number on the column: the one rating it sits in gives it 4, the other 2 give 0, and 4 / 3 is 1.3333"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 0, 0), -0.33333333333333326,
         'customer_slope(customers, films, ratings, 0, 0)',
         "Ida's first number on the whole table: the 2 ratings it sits in give it 10 and -12, the other 4 give 0, and the sum divided by all 6 ratings is -0.3333"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 0, 1), -4.333333333333334,
         'customer_slope(customers, films, ratings, 0, 1)',
         "Ida's second number on the whole table: the 2 ratings it sits in give it -20 and -6, the other 4 give 0, and the sum divided by all 6 ratings is -4.3333"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 1, 0), -2.333333333333333,
         'customer_slope(customers, films, ratings, 1, 0)',
         "Raj's first number on the whole table: the 2 ratings it sits in give it -6 and -8, the other 4 give 0, and the sum divided by all 6 ratings is -2.3333"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 1, 1), 1.3333333333333335,
         'customer_slope(customers, films, ratings, 1, 1)',
         "Raj's second number on the whole table: the 2 ratings it sits in give it 12 and -4, the other 4 give 0, and the sum divided by all 6 ratings is 1.3333"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 2, 0), 2.333333333333333,
         'customer_slope(customers, films, ratings, 2, 0)',
         "Elsa's first number on the whole table: the 2 ratings it sits in give it -2 and 16, the other 4 give 0, and the sum divided by all 6 ratings is 2.3333"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 2, 1), 2.0,
         'customer_slope(customers, films, ratings, 2, 1)',
         "Elsa's second number on the whole table: the 2 ratings it sits in give it 4 and 8, the other 4 give 0, and the sum divided by all 6 ratings is 2"),
        (([[-2, 0, 2], [2, -2, -1], [2, 2, 2]], [[1, 1, -1, -2], [2, 2, -2, -2], [2, 1, 2, 0]], [[7, 0, -7, -6], [-4, -6, 10, 0], [-1, 7, -2, 2]], 1, 2), -2.8333333333333335,
         'customer_slope(score_customers, score_films, scores, 1, 2)',
         "Leo's third number on part 1's scores: the 4 ratings it sits in give it 0, 6, -40 and 0, the other 8 give 0, and the sum divided by all 12 ratings is -2.8333"),
        (([[-2, 0, 2], [2, -2, -1], [2, 2, 2]], [[1, 1, -1, -2], [2, 2, -2, -2], [2, 1, 2, 0]], [[7, 0, -7, -6], [-4, -6, 10, 0], [-1, 7, -2, 2]], 2, 0), 5.333333333333334,
         'customer_slope(score_customers, score_films, scores, 2, 0)',
         "Maya's first number on part 1's scores: the 4 ratings it sits in give it 22, 2, 0 and 40, the other 8 give 0, and the sum divided by all 12 ratings is 5.3333"),
        (([[-0.9333333333333333, -1.1333333333333333], [-1.5333333333333334, -0.2666666666666667], [1.5333333333333334, 1.6]], [[-1.0666666666666667, -1.0], [1.2, -0.06666666666666676]], [[2, 1], [-1, 2], [1, -2]], 0, 1), -0.9459753086419753,
         'customer_slope(customers, films, ratings (one step after the first guess), 0, 1)',
         "Ida's second number on the table after one step: the 2 ratings it sits in give it -5.6747 and -0.0012, the other 4 give 0, and the sum divided by all 6 ratings is -0.946"),
    ],
    "film_slope": [
        (([[-1, -2], [-2, 0], [2, 2]], [[-1], [2]], [[2], [-1], [1]], 0, 0), 0.6666666666666667,
         'film_slope(customers, column_films, column_ratings, 0, 0)',
         "Paddington's first number on the column: the 3 ratings give it 10, -12 and 4, and their average is 0.6667"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1], [2]], [[2], [-1], [1]], 0, 1), 8.0,
         'film_slope(customers, column_films, column_ratings, 0, 1)',
         "Paddington's second number on the column: the 3 ratings give it 20, 0 and 4, and their average is 8"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 0, 0), 0.33333333333333337,
         'film_slope(customers, films, ratings, 0, 0)',
         "Paddington's first number on the whole table: the 3 ratings it sits in give it 10, -12 and 4, the other 3 give 0, and the sum divided by all 6 ratings is 0.3333"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 0, 1), 4.0,
         'film_slope(customers, films, ratings, 0, 1)',
         "Paddington's second number on the whole table: the 3 ratings it sits in give it 20, 0 and 4, the other 3 give 0, and the sum divided by all 6 ratings is 4"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 1, 0), -5.0,
         'film_slope(customers, films, ratings, 1, 0)',
         "The Martian's first number on the whole table: the 3 ratings it sits in give it -6, -8 and -16, the other 3 give 0, and the sum divided by all 6 ratings is -5"),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 1, 1), -4.666666666666666,
         'film_slope(customers, films, ratings, 1, 1)',
         "The Martian's second number on the whole table: the 3 ratings it sits in give it -12, 0 and -16, the other 3 give 0, and the sum divided by all 6 ratings is -4.6667"),
        (([[-2, 0, 2], [2, -2, -1], [2, 2, 2]], [[1, 1, -1, -2], [2, 2, -2, -2], [2, 1, 2, 0]], [[7, 0, -7, -6], [-4, -6, 10, 0], [-1, 7, -2, 2]], 0, 0), 5.333333333333333,
         'film_slope(score_customers, score_films, scores, 0, 0)',
         "The Intouchables's first number on part 1's scores: the 3 ratings it sits in give it 20, 0 and 44, the other 9 give 0, and the sum divided by all 12 ratings is 5.3333"),
        (([[-2, 0, 2], [2, -2, -1], [2, 2, 2]], [[1, 1, -1, -2], [2, 2, -2, -2], [2, 1, 2, 0]], [[7, 0, -7, -6], [-4, -6, 10, 0], [-1, 7, -2, 2]], 3, 2), 0.0,
         'film_slope(score_customers, score_films, scores, 3, 2)',
         "Interstellar's third number on part 1's scores: the 3 ratings it sits in give it 40, 0 and -40, the other 9 give 0, and the sum divided by all 12 ratings is 0"),
        (([[-0.9333333333333333, -1.1333333333333333], [-1.5333333333333334, -0.2666666666666667], [1.5333333333333334, 1.6]], [[-1.0666666666666667, -1.0], [1.2, -0.06666666666666676]], [[2, 1], [-1, 2], [1, -2]], 1, 1), 0.22854320987654303,
         'film_slope(customers, films, ratings (one step after the first guess), 1, 1)',
         "The Martian's second number on the table after one step: the 3 ratings it sits in give it -0.0201, 0.2394 and 1.152, the other 3 give 0, and the sum divided by all 6 ratings is 0.2285"),
    ],
    "table_step": [
        (([[-1, -2], [-2, 0], [2, 2]], [[-1], [2]], [[2], [-1], [1]], 0.2), (4.594008230452675, [[-1.6666666666666667, -0.6666666666666665], [-1.6, -0.8], [2.1333333333333333, 1.7333333333333334]], [[-1.1333333333333333], [0.3999999999999999]]),
         'table_step(customers, column_films, column_ratings, 0.2) from the first guess',
         'One step on the column: Ida -1.67 and -0.67; Raj -1.6 and -0.8; Elsa 2.13 and 1.73, Paddington -1.13 and 0.4, and the average loss 4.594'),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 0.2), (1.9659325102880663, [[-0.9333333333333333, -1.1333333333333333], [-1.5333333333333334, -0.2666666666666667], [1.5333333333333334, 1.6]], [[-1.0666666666666667, -1.0], [1.2, -0.06666666666666676]]),
         'table_step(customers, films, ratings, 0.2) from the first guess',
         'One step on the whole table: the average falls from 10.6667 to 1.9659'),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 0.1), (3.9826547325102886, [[-0.9666666666666667, -1.5666666666666667], [-1.7666666666666666, -0.13333333333333336], [1.7666666666666666, 1.8]], [[-1.0333333333333334, -1.5], [1.6, -0.5333333333333334]]),
         'table_step(customers, films, ratings, 0.1) from the first guess',
         'Another step size, 0.1: the average falls from 10.6667 to 3.9827'),
        (([[-2, 0, 2], [2, -2, -1], [2, 2, 2]], [[1, 1, -1, -2], [2, 2, -2, -2], [2, 1, 2, 0]], [[7, 0, -7, -6], [-4, -6, 10, 0], [-1, 7, -2, 2]], 0.2), (9.056290329218106, [[-0.7333333333333332, 1.866666666666667, 1.4666666666666668], [1.5666666666666667, -2.8666666666666667, -0.43333333333333324], [0.9333333333333331, 0.5333333333333332, 1.2333333333333334]], [[-0.06666666666666665, 0.7333333333333334, 0.5333333333333332, -0.6666666666666665], [1.2666666666666666, 2.1333333333333333, -2.666666666666667, -1.3333333333333333], [1.6, 1.0333333333333334, 0.7999999999999998, 0.0]]),
         'table_step(score_customers, score_films, scores, 0.2)',
         "Part 1's scores, three numbers each: the average falls from 52.0833 to 9.0563"),
        (([[2, 1], [-1, 2], [1, -2]], [[1, 0], [0, 1]], [[2, 1], [-1, 2], [1, -2]], 0.2), (0.0, [[2.0, 1.0], [-1.0, 2.0], [1.0, -2.0]], [[1.0, 0.0], [0.0, 1.0]]),
         'table_step(ratings, [[1, 0], [0, 1]], ratings, 0.2)',
         'Every rating exact: every slope is 0, so no number moves, and the loss stays 0'),
        (([[-0.9333333333333333, -1.1333333333333333], [-1.5333333333333334, -0.2666666666666667], [1.5333333333333334, 1.6]], [[-1.0666666666666667, -1.0], [1.2, -0.06666666666666676]], [[2, 1], [-1, 2], [1, -2]], 0.2), (1.3191707824452246, [[-1.100879012345679, -0.9441382716049382], [-1.3985975308641976, -0.4539061728395062], [1.5064493827160494, 1.6588444444444446]], [[-0.9039407407407407, -1.0821333333333332], [1.1388444444444445, -0.11237530864197537]]),
         'table_step(customers, films, ratings, 0.2), the second step',
         'The second step on the whole table: the average falls from 1.9659 to 1.3192'),
    ],
    "factorize": [
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 0.2, 1000), ([10.666666666666666, 1.9659325102880663, 1.3191707824452246, 1.0213962411298187, 0.8283397232580261, 0.6788029082308977, 0.5522205953926653, 0.44171995348854165, 0.3458399642257273, 0.26486458505465066, 0.19893455305532584, 0.14726275636406647, 0.10812580167310831, 0.079267665578269, 0.058368705071382615, 0.04337327914380757, 0.03263095964025954, 0.02490093030600239, 0.01928868408776328, 0.015165100707456497, 0.012093147647627499, 0.009770405707680488, 0.007987240643662831, 0.006597552611807593, 0.005498729909862848, 0.004618026727832565, 0.0039033212394153707, 0.0033168313802905427, 0.0028308233186916214, 0.002424665020480633, 0.0020827913312590463, 0.0017932896971076344, 0.001546910583008092, 0.0013363699676184973, 0.0011558537197122762, 0.00100066223819379, 0.000866953083335196, 0.0007515524834703185, 0.0006518155845946985, 0.0005655214653134361, 0.000490793169885075, 0.0004260359280914033, 0.0003698887465675995, 0.00032118595425634173, 0.0002789262576103557, 0.00024224754109564172, 0.00021040612602541485, 0.00018275953802562826, 0.00015875207332713298, 0.0001379026261123955, 0.00011979436368892305, 0.00010406592740517115, 9.040390473291215e-05], [[-1.354970032185723, 0.34890132512953625], [-1.3229004707977505, -1.6007709663492258], [1.3487838256379492, 1.620634918420078]], [[-1.0845583259953036, -0.871722983612972], [1.520322023266681, -0.5186161242852434]]),
         'factorize(customers, films, ratings, 0.2, 1000) from the first guess',
         'The whole table: the average loss is below 0.0001 after 52 steps: 10.6667, 1.9659, 1.3192, 1.0214, 0.8283, ..., 0.00012, 0.0001, about 0'),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1], [2]], [[2], [-1], [1]], 0.2, 1000), ([11.666666666666666, 4.594008230452675, 3.7437476548574327, 2.0064880450088762, 0.7370914670837904, 0.25708931499345916, 0.07936525543271926, 0.02225398488154527, 0.006742072833706742, 0.0024228188594692173, 0.000983468731877809, 0.0004218704581360469, 0.0001849040601954538, 8.172620650345672e-05], [[-1.9309535210950082, 0.268913061599475], [-0.8342201803391485, -1.2752259073688792], [1.4832041851995288, 1.7064889595755544]], [[-0.8513319616379365], [1.3306142283355162]]),
         'factorize(customers, column_films, column_ratings, 0.2, 1000) from the first guess',
         'The column: below 0.0001 after 13 steps'),
        (([[2, 1], [-1, 2], [1, -2]], [[1, 0], [0, 1]], [[2, 1], [-1, 2], [1, -2]], 0.2, 1000), ([0.0], [[2, 1], [-1, 2], [1, -2]], [[1, 0], [0, 1]]),
         'factorize(ratings, [[1, 0], [0, 1]], ratings, 0.2, 1000)',
         'This guess gives every rating exactly: the loss at the start is 0, so the loop makes no step, and the list holds one loss'),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]], [[2, 1], [-1, 2], [1, -2]], 0.2, 5), ([10.666666666666666, 1.9659325102880663, 1.3191707824452246, 1.0213962411298187, 0.8283397232580261, 0.6788029082308977], [[-1.3272972816720454, -0.5143880196995813], [-1.2395752563569975, -0.8010583994307836], [1.4690820751398717, 1.7479826151629567]], [[-0.8319779930229826, -1.136707074522835], [1.139044524050882, -0.14340698798664636]]),
         'factorize(customers, films, ratings, 0.2, 5)',
         'At most 5 steps: after them the loss, 0.6788, is still above 0.0001, so the list holds 6 losses'),
        (([[-1, -2], [-2, 0], [2, 2]], [[-1], [2]], [[2], [-1], [1]], 0.1, 1000), ([11.666666666666666, 3.8754897119341565, 2.471959779898967, 1.8209070153686782, 1.3749526198150788, 1.0256612756040568, 0.7399117801895163, 0.5093601626291598, 0.33228522916903286, 0.20523084373389303, 0.12057933650619777, 0.06801535334422258, 0.03727451172814059, 0.020097091674891012, 0.010782922305583134, 0.005810357414707226, 0.0031644594664110557, 0.0017481914401060493, 0.0009808676643007464, 0.0005586685288623556, 0.0003225160719047635, 0.00018833031201531455, 0.0001110051315899791, 6.591312256399353e-05], [[-2.164224111503711, -0.21708171322153666], [-1.0315911888303126, -1.4231038173520232], [1.9058761541066116, 2.072786142764268]], [[-1.0709570265169202], [1.4703363147093875]]),
         'factorize(customers, column_films, column_ratings, 0.1, 1000)',
         'Another step size, 0.1: the column is below 0.0001 after 23 steps'),
        (([[-2, 0, 2], [2, -2, -1], [2, 2, 2]], [[1, 1, -1, -2], [2, 2, -2, -2], [2, 1, 2, 0]], [[7, 0, -7, -6], [-4, -6, 10, 0], [-1, 7, -2, 2]], 0.2, 1000), ([52.083333333333336, 9.056290329218106, 7.459148948015283, 6.983488889858573, 6.585994494826732, 6.086702684377696, 5.428361172559723, 4.596478977482215, 3.632855400953215, 2.647309221992773, 1.784286273310437, 1.1448086206086194, 0.7351005219730281, 0.49513314403629266, 0.35783524210460044, 0.2772616655856662, 0.22750920653705722, 0.19476445315134952, 0.17162299515409515, 0.15404095489574426, 0.13977945622846868, 0.12759169517817762, 0.11678325780913319, 0.10696846438565065, 0.09793355334024197, 0.0895594361882132, 0.08177807765369358, 0.07454797834674813, 0.0678405472681274, 0.06163269208338796, 0.055902957052129026, 0.050629675197167316, 0.04579024912423205, 0.041361046036134864, 0.03731760680940321, 0.03363499418305328, 0.030288179267828192, 0.027252410175547536, 0.024503533701057095, 0.022018257559506513, 0.019774350688550175, 0.017750784955321782, 0.015927824740195294, 0.014287072256666852, 0.012811476745302338, 0.011485315283558041, 0.010294152180114303, 0.009224782974058814, 0.008265168068696416, 0.007404360079919605, 0.00663242811731802, 0.0059403814651820125, 0.0053200944964404105, 0.004764234131212696, 0.00426619073299231, 0.0038200130066305786, 0.0034203472095875377, 0.0030623807981575066, 0.002741790491550348, 0.002454694638310535, 0.002197609702684315, 0.001967410645836243, 0.0017612949522991426, 0.0015767500409545531, 0.0014115237984500704, 0.0012635979783744364, 0.0011311642195261902, 0.001012602449566295, 0.000906461455012228, 0.0008114414140101793, 0.000726378203978611, 0.0006502293115994213, 0.000582061187447307, 0.0005210379016009163, 0.0004664109697592713, 0.00041751023164180194, 0.0003737356747641463, 0.00033455010707192373, 0.00029947259141253104, 0.00026807256347706373, 0.00023996456270304904, 0.00021480351274889744, 0.00019228049459011262, 0.00017211896110039192, 0.00015407134722232393, 0.0001379160345523669, 0.00012345463341197305, 0.00011050954929475421, 9.892180401018559e-05], [[-2.6375397725555763, 1.9839206912684235, 1.6165192883283697], [0.2685407300646917, -3.006391466533088, -0.043545365523291996], [2.0276858543533476, 0.892638798194137, 0.7435787459040232]], [[-1.2912163023266676, 2.17351359610601, 0.3576203145411484, 1.471923595086661], [1.198689362087395, 2.1768375602531185, -3.301153109416537, 0.14940901976830026], [0.7602735739271915, 0.875298250756269, 0.30808881052865633, -1.4882031150572415]]),
         'factorize(score_customers, score_films, scores, 0.2, 1000)',
         "Part 1's scores, three numbers each: below 0.0001 after 88 steps"),
    ],
}

_P4_SUCCESS = {'average_loss': 'The column gives about 11.6667 and the whole table about 10.6667.', 'customer_slope': "On the column Ida's first number gets 3.3333.", 'film_slope': "On the column Paddington's first number gets 0.6667 and its second 8.", 'table_step': 'One step takes the column from about 11.6667 to 4.594, and the table from 10.6667 to 1.9659.', 'factorize': 'The whole table is below 0.0001 after 52 steps.'}


class _P4_Stop(Exception):
    """Raised by the recorder of check_factorize to stop a call that cannot go on."""


def _p4_call(fn, inputs):
    """Run a participant's function on fresh copies of the inputs, with its prints silenced. Gives back (answer,
    error, the copies after the call); error is None when the call went through. Never raises."""
    import contextlib, copy, io
    copies = copy.deepcopy(list(inputs))
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*copies)
        return answer, None, copies
    except _P4_Stop as stop:
        return None, "STOP " + str(stop), copies
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error), copies


def _p4_close(a, b):
    """True when a and b are numbers that agree within the margin (relative, and absolute near 0)."""
    if not _p4_is_number(a) or not _p4_is_number(b):
        return False
    try:
        return abs(a - b) <= _P4_MARGIN * max(1.0, abs(b))
    except Exception:
        return False


def _p4_close_matrix(a, b):
    """True when a and b are matrices of the same shape whose numbers agree within the margin."""
    if not _p4_matrix(a) or not _p4_matrix(b) or len(a) != len(b) or len(a[0]) != len(b[0]):
        return False
    for i in range(len(a)):
        for k in range(len(a[0])):
            if not _p4_close(a[i][k], b[i][k]):
                return False
    return True


def _p4_close_list(a, b):
    """True when a and b are lists of numbers of the same length that agree within the margin."""
    if not isinstance(a, list) or not isinstance(b, list) or len(a) != len(b):
        return False
    for k in range(len(a)):
        if not _p4_close(a[k], b[k]):
            return False
    return True


def _p4_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    # A gap such as "x = ..." stays in the compiled function; a gap that stands on its own line is only
    # visible in the source, which Jupyter and Colab keep for every cell
    code = getattr(fn, "__code__", None)
    if code is not None and Ellipsis in code.co_consts:
        return True
    import inspect
    try:
        lines = inspect.getsource(fn).splitlines()
    except Exception:
        return False
    for line in lines:
        if "..." in line.split("#")[0]:
            return True
    return False


def _p4_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _p4_matrix_text(m):
    """[[-1.6667, -0.6667], [-1.6, -0.8]], with at most 4 decimals, cut short when it is long."""
    if not _p4_matrix(m):
        return _p4_short(m)
    rows = []
    for row in m:
        texts = []
        for x in row:
            texts.append(_p4_dec(x, 4))
        rows.append("[" + ", ".join(texts) + "]")
    text = "[" + ", ".join(rows) + "]"
    if len(text) > 110:
        text = text[:107] + "..."
    return text


def _p4_list_text(values):
    """[10.6667, 1.9659, ...], with at most 6 decimals, cut short when it is long."""
    if not isinstance(values, list):
        return _p4_short(values)
    texts = []
    for x in values:
        if _p4_is_number(x):
            texts.append(_p4_dec(x, 6))
        else:
            texts.append(_p4_short(x))
    if len(texts) > 9:
        texts = texts[:5] + ["..."] + texts[-3:]
    return "[" + ", ".join(texts) + "]"


def _p4_error_hint(error, name):
    """A hint for an error that stopped a participant's function."""
    if "UnboundLocalError" in error or "before assignment" in error or "not callable" in error:
        return ("a variable and a function share a name. Inside a function, average_loss = ... makes average_loss "
                "a number, and Python forgets the function there. Give the variable another name, for example "
                "loss. If you typed such a line in a cell, run the cell of the task again to bring the function "
                "back, and the cell with part 1's functions for inner_product or column.")
    if "NameError" in error:
        return ("Python does not know a name that %s uses. Run the setup cell of part 4, the cell with part 1's "
                "functions, and the cells of the tasks above, then the cell of %s, then this check." % (name, name))
    if "ZeroDivisionError" in error:
        if name == "average_loss":
            return "count is still 0: count = count + 1 belongs inside both loops, once for every rating."
        return ("divide by the number of all the ratings of the table, count = len(customers) * len(films[0]): "
                "films has one column per film.")
    if "IndexError" in error:
        if name == "average_loss":
            return ("films has one row per number and one column per film: film j's numbers are its column, "
                    "column(films, j), and len(films[0]) counts the films. customers[i] is the row of customer i, "
                    "and ratings[i][j] is the rating of customer i for film j.")
        if name == "customer_slope":
            return ("customers[i][position] sits in the predictions of customer i, one for every film: loop over "
                    "the films, for j in range(len(films[0])), with the partner films[position][j], film j's number "
                    "at the same position (films has one row per number and one column per film), and the rating "
                    "ratings[i][j].")
        if name == "film_slope":
            return ("films[position][j] sits in the predictions of film j, one for every customer: loop over the "
                    "customers, for i in range(len(customers)), with the partner customers[i][position] and the "
                    "rating ratings[i][j].")
        return ("customers has one row per customer, films one row per number and one column per film, and every "
                "slope sits in the place of its number: customer_slopes[i][position] belongs to "
                "customers[i][position], and film_slopes[position][j] to films[position][j], film j's number at "
                "that position. film_slope takes the film first: film_slope(customers, films, ratings, j, "
                "position).")
    if name == "average_loss":
        return ("inner_product takes two lists of numbers, customers[i] and column(films, j), and gives back one "
                "number.")
    if name in ("customer_slope", "film_slope"):
        return ("the partner is one number, films[position][j] for a customer's number and customers[i][position] "
                "for a film's number, and the difference is prediction - ratings[i][j].")
    if name == "table_step":
        return ("customer_slope(customers, films, ratings, i, position) and film_slope(customers, films, ratings, "
                "j, position) take the whole guess and the place of one number, and give back one number.")
    return ("table_step(customers, films, ratings, alpha) takes the whole guess and gives back one number, the "
            "average loss after the step: loss = table_step(customers, films, ratings, alpha), then "
            "losses.append(loss).")


def _p4_slip(fn, name, slips):
    """The hint of the first slip whose answers fn gives on every test of name, or None. A test on which both
    the slip and fn stop with an error counts as the same answer."""
    for slip, hint in slips:
        same = True
        for inputs, expected, call, why in _P4_TESTS[name]:
            answer, error, after = _p4_call(fn, inputs)
            try:
                wanted = slip(*_p4_call(lambda *x: x, inputs)[0])
                slip_failed = False
            except Exception:
                slip_failed = True
            if slip_failed:
                if error is None:
                    same = False
                    break
                continue
            if error is not None or not _p4_close(answer, wanted):
                same = False
                break
        if same:
            return hint
    return None


# Task A, average_loss

def _p4_squares(c, f, r):
    """The loss of every rating, in reading order."""
    out = []
    for i in range(len(c)):
        for j in range(len(f[0])):
            difference = _p4_inner(c[i], _p4_column(f, j)) - r[i][j]
            out.append(difference * difference)
    return out


def _p4_differences(c, f, r):
    out = []
    for i in range(len(c)):
        for j in range(len(f[0])):
            out.append(_p4_inner(c[i], _p4_column(f, j)) - r[i][j])
    return out


def _p4_by_rows(c, f, r):
    """The loss of every rating with film j's numbers read from row j of films, the old shape (one row per film):
    films[j] in place of column(films, j). It stops with an error where such a function stops."""
    out = []
    for i in range(len(c)):
        for j in range(len(f[0])):
            difference = _p4_inner(c[i], f[j]) - r[i][j]
            out.append(difference * difference)
    return out


def _p4_loss_slips():
    S = _p4_squares
    D = _p4_differences
    return [
        (lambda c, f, r: sum(S(c, f, r)),
         "this is the sum of the losses. The average divides it by the number of ratings: return total / count."),
        (lambda c, f, r: sum(S(c, f, r)) / len(c),
         "you divided by the number of customers. The average divides by the number of all the ratings, one for "
         "every customer and every film: count goes up by 1 for each of them."),
        (lambda c, f, r: sum(S(c, f, r)) / len(f[0]),
         "you divided by the number of films. The average divides by the number of all the ratings, one for every "
         "customer and every film: count goes up by 1 for each of them."),
        (lambda c, f, r: sum(S(c, f, r)) / (len(c) * len(f)),
         "len(films) counts the rows of films, one per number, and the films are its columns, len(films[0]). The "
         "average divides by the number of all the ratings, one for every customer and every film: count goes up "
         "by 1 for each of them."),
        (lambda c, f, r: sum(_p4_by_rows(c, f, r)) / (len(c) * len(f[0])),
         "films[j] is row j of films, the numbers at position j of every film. films has one row per number and "
         "one column per film, so film j's numbers stand in its column: prediction = "
         "inner_product(customers[i], column(films, j))."),
        (lambda c, f, r: sum(D(c, f, r)) / len(D(c, f, r)),
         "the differences were added without squaring them. The loss of a rating is the difference times itself: "
         "total = total + difference * difference."),
        (lambda c, f, r: sum(abs(x) for x in D(c, f, r)) / len(D(c, f, r)),
         "the loss of a rating is the difference squared, difference * difference, so a miss twice as big costs "
         "four times as much."),
        (lambda c, f, r: S(c, f, r)[-1] / len(S(c, f, r)),
         "only the last rating's loss is in total. Write total = total + difference * difference, so that every "
         "rating adds its loss to the ones before."),
        (lambda c, f, r: S(c, f, r)[0],
         "the function returned after the first rating: the return line sits inside the loops. Move it out of "
         "both loops, to the level of total = 0, so that it comes after every rating."),
        (lambda c, f, r: sum(S(c, f, r)[:len(f[0])]) / len(f[0]),
         "the function returned after the first customer's ratings: the return line sits inside the loop over the "
         "customers. Move it out of both loops, to the level of total = 0."),
    ]


def _p4_run_average_loss(fn):
    """None when average_loss passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P4_TESTS["average_loss"]:
        answer, error, copies = _p4_call(fn, inputs)
        if error is not None:
            hint = _p4_slip(fn, "average_loss", _p4_loss_slips())
            return ("%s stopped with an error: %s" % (call, error), hint or _p4_error_hint(error, "average_loss"))
        for k in range(3):
            if not _p4_close_matrix(copies[k], inputs[k]):
                return ("%s changed the lists it got" % call,
                        "average_loss only measures: it reads customers, films and ratings and leaves them as they "
                        "are.")
        if _p4_close(answer, expected):
            continue
        if answer is None:
            return ("%s gave no answer (None). Does the function end with a return line?" % call,
                    "the last line returns the average: return total / count.")
        if not _p4_is_number(answer):
            return ("%s gave %s, expected %s. %s." % (call, _p4_short(answer), _p4_dec(expected, 6), why),
                    "return one number, the average of the losses: total / count.")
        hint = _p4_slip(fn, "average_loss", _p4_loss_slips())
        if hint is None:
            hint = ("for every rating: prediction = inner_product(customers[i], column(films, j)), difference = "
                    "prediction - ratings[i][j], total = total + difference * difference; after both loops, "
                    "return total / count.")
        return ("%s gave %s, expected %s. %s." % (call, _p4_dec(answer, 6), _p4_dec(expected, 6), why), hint)
    return None


# Tasks B and C, customer_slope and film_slope

def _p4_d(c, f, r, i, j):
    """The difference of a rating: the prediction minus the rating."""
    return _p4_inner(c[i], _p4_column(f, j)) - r[i][j]


_P4_OLD_COUNT = ("len(films) counts the rows of films, one per number, and the films are its columns: count = "
                 "len(customers) * len(films[0]), the number of ALL the ratings of the table.")


def _p4_customer_slips():
    d = _p4_d

    def total(c, f, r, i, p, factor=2, partner=None, which=None):
        out = 0
        films = range(len(f[0])) if which is None else which(f)
        for j in films:
            mate = f[p][j] if partner is None else partner(c, f, i, j, p)
            out = out + factor * mate * d(c, f, r, i, j)
        return out

    def n(c, f):
        return len(c) * len(f[0])
    return [
        (lambda c, f, r, i, p: total(c, f, r, i, p) / len(f[0]),
         "you divided by the number of ratings the number sits in, one for every film. Divide by the number of "
         "ALL the ratings of the table, len(customers) * len(films[0]): the ratings of the other customers give "
         "the number 0, and they still count in the average."),
        (lambda c, f, r, i, p: total(c, f, r, i, p) / len(c),
         "you divided by the number of customers. Divide by the number of ALL the ratings of the table, "
         "len(customers) * len(films[0])."),
        (lambda c, f, r, i, p: total(c, f, r, i, p),
         "this is the sum of what the ratings give. The slope of the average loss divides it by the number of all "
         "the ratings: return total / count, with count = len(customers) * len(films[0])."),
        (lambda c, f, r, i, p: total(c, f, r, i, p) / (len(c) * len(f)), _P4_OLD_COUNT),
        (lambda c, f, r, i, p: total(c, f, r, i, p, partner=lambda c, f, i, j, p: c[i][p]) / n(c, f),
         "you used the number itself, customers[i][position]. The rule takes its partner, the number it meets in "
         "the prediction: films[position][j], film j's number at the same position."),
        (lambda c, f, r, i, p: total(c, f, r, i, p, partner=lambda c, f, i, j, p: f[j][p]) / n(c, f),
         "films[j][position] stands in row j and column position of films. films has one row per number and one "
         "column per film, so the partner, film j's number at the same position, is films[position][j]."),
        (lambda c, f, r, i, p: total(c, f, r, i, p, factor=1) / n(c, f),
         "the 2 is missing, so your slopes are half as big: each rating gives 2 * partner * difference."),
        (lambda c, f, r, i, p: total(c, f, r, i, p, factor=-2) / n(c, f),
         "the sign is flipped. The difference is the prediction minus the rating, prediction - ratings[i][j], and "
         "each rating gives 2 * partner * difference."),
        (lambda c, f, r, i, p: total(c, f, r, i, p, factor=-1) / n(c, f),
         "the sign is flipped and the 2 is missing: each rating gives 2 * partner * difference, with the "
         "difference prediction - ratings[i][j]."),
        (lambda c, f, r, i, p: total(c, f, r, i, p, partner=lambda c, f, i, j, p: f[1 - p][j]) / n(c, f),
         "the partner sits at the same position in the film's column: films[position][j]."),
        (lambda c, f, r, i, p: total(c, f, r, i, p, which=lambda f: [len(f[0]) - 1]) / n(c, f),
         "only the last film's rating counted. Write total = total + ..., so that every film's rating adds its "
         "share: your code kept only the last one."),
        (lambda c, f, r, i, p: total(c, f, r, i, p, which=lambda f: [0]) / n(c, f),
         "only the first film's rating counted: the return line sits inside the loop. Move it out of the loop, "
         "so that every film's rating adds its share first."),
    ]


def _p4_film_slips():
    d = _p4_d

    def total(c, f, r, j, p, factor=2, partner=None, which=None):
        out = 0
        customers = range(len(c)) if which is None else which(c, f)
        for i in customers:
            mate = c[i][p] if partner is None else partner(c, f, i, j, p)
            out = out + factor * mate * d(c, f, r, i, j)
        return out

    def n(c, f):
        return len(c) * len(f[0])
    return [
        (lambda c, f, r, j, p: total(c, f, r, j, p) / len(c),
         "you divided by the number of ratings the number sits in, one for every customer. Divide by the number "
         "of ALL the ratings of the table, len(customers) * len(films[0]): the ratings of the other films give the "
         "number 0, and they still count in the average."),
        (lambda c, f, r, j, p: total(c, f, r, j, p) / len(f[0]),
         "you divided by the number of films. Divide by the number of ALL the ratings of the table, "
         "len(customers) * len(films[0])."),
        (lambda c, f, r, j, p: total(c, f, r, j, p),
         "this is the sum of what the ratings give. The slope of the average loss divides it by the number of all "
         "the ratings: return total / count, with count = len(customers) * len(films[0])."),
        (lambda c, f, r, j, p: total(c, f, r, j, p) / (len(c) * len(f)), _P4_OLD_COUNT),
        (lambda c, f, r, j, p: total(c, f, r, j, p, partner=lambda c, f, i, j, p: f[p][j]) / n(c, f),
         "you used the number itself, films[position][j]. The rule takes its partner, the number it meets in the "
         "prediction: customers[i][position], customer i's number at the same position."),
        (lambda c, f, r, j, p: total(c, f, r, j, p, factor=1) / n(c, f),
         "the 2 is missing, so your slopes are half as big: each rating gives 2 * partner * difference."),
        (lambda c, f, r, j, p: total(c, f, r, j, p, factor=-2) / n(c, f),
         "the sign is flipped. The difference is the prediction minus the rating, prediction - ratings[i][j], and "
         "each rating gives 2 * partner * difference."),
        (lambda c, f, r, j, p: total(c, f, r, j, p, factor=-1) / n(c, f),
         "the sign is flipped and the 2 is missing: each rating gives 2 * partner * difference, with the "
         "difference prediction - ratings[i][j]."),
        (lambda c, f, r, j, p: total(c, f, r, j, p, partner=lambda c, f, i, j, p: c[i][1 - p]) / n(c, f),
         "the partner sits at the same position in the customer's row: customers[i][position]."),
        (lambda c, f, r, j, p: total(c, f, r, j, p, which=lambda c, f: range(len(f[0]))) / n(c, f),
         "you looped over the films. A film's number sits in the predictions of every customer: for i in "
         "range(len(customers))."),
        (lambda c, f, r, j, p: total(c, f, r, j, p, which=lambda c, f: [len(c) - 1]) / n(c, f),
         "only the last customer's rating counted. Write total = total + ..., so that every customer's rating "
         "adds its share: your code kept only the last one."),
        (lambda c, f, r, j, p: total(c, f, r, j, p, which=lambda c, f: [0]) / n(c, f),
         "only the first customer's rating counted: the return line sits inside the loop. Move it out of the "
         "loop, so that every customer's rating adds its share first."),
    ]


def _p4_run_slope(fn, name):
    """None when customer_slope or film_slope passes every test, else (what went wrong, hint)."""
    if name == "customer_slope":
        loop, partner, slips = "the films j", "films[position][j]", _p4_customer_slips()
    else:
        loop, partner, slips = "the customers i", "customers[i][position]", _p4_film_slips()
    for inputs, expected, call, why in _P4_TESTS[name]:
        answer, error, copies = _p4_call(fn, inputs)
        if error is not None:
            hint = _p4_slip(fn, name, slips)
            return ("%s stopped with an error: %s" % (call, error), hint or _p4_error_hint(error, name))
        for k in range(3):
            if not _p4_close_matrix(copies[k], inputs[k]):
                return ("%s changed the lists it got" % call,
                        "%s only measures: it reads customers, films and ratings and leaves them as they are."
                        % name)
        if _p4_close(answer, expected):
            continue
        if answer is None:
            return ("%s gave no answer (None). Does the function end with a return line?" % call,
                    "the last line returns the average over all the ratings: return total / count.")
        if not _p4_is_number(answer):
            return ("%s gave %s, expected %s. %s." % (call, _p4_short(answer), _p4_dec(expected, 6), why),
                    "return one number: what the ratings give the number, added up and divided by the number of "
                    "all the ratings.")
        hint = _p4_slip(fn, name, slips)
        if hint is None:
            hint = ("loop over %s: difference = prediction - ratings[i][j], partner = %s, total = total + 2 * "
                    "partner * difference; then return total / count, with count = len(customers) * len(films[0])."
                    % (loop, partner))
        return ("%s gave %s, expected %s. %s." % (call, _p4_dec(answer, 6), _p4_dec(expected, 6), why), hint)
    return None


# Task D, table_step

_P4_STEP_SLIPS = [
    ("one at a time",
     "close, but every slope must be measured on the guess as it was before the step. Measure every slope first, "
     "in the first loops, and move the numbers in the last loops with the slopes you measured: "
     "customers[i][position] - alpha * customer_slopes[i][position]."),
    ("customers first",
     "close, but the films' slopes were measured after the customers had moved. Measure every slope first, the "
     "customers' and the films', and only then move the numbers."),
    ("plus",
     "the numbers moved with their slopes, uphill, and the loss grew. Move against the slope, with a minus: "
     "customers[i][position] - alpha * customer_slopes[i][position]."),
    ("no alpha",
     "every number lost its whole slope, a giant jump. The step is alpha times the slope: "
     "customers[i][position] - alpha * customer_slopes[i][position]."),
    ("divided",
     "the slope was divided by alpha, a giant jump. The step is alpha times the slope: "
     "customers[i][position] - alpha * customer_slopes[i][position]."),
    ("only customers",
     "the customers moved right, and the films did not move. In the last loop, set films[position][j] = "
     "films[position][j] - alpha * film_slopes[position][j], on the list that the function got: a line films = "
     "... inside the function makes a new list, and the old one stays as it was."),
    ("only films",
     "the films moved right, and the customers did not move. In the loop over the customers, set "
     "customers[i][position] = customers[i][position] - alpha * customer_slopes[i][position], on the list that "
     "the function got: a line customers = ... inside the function makes a new list, and the old one stays as it "
     "was."),
]


def _p4_run_table_step(fn):
    """None when table_step passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P4_TESTS["table_step"]:
        customers, films, ratings, alpha = inputs
        expected_loss, expected_customers, expected_films = expected
        answer, error, copies = _p4_call(fn, inputs)
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p4_error_hint(error, "table_step"))
        after_c, after_f = copies[0], copies[1]
        wanted = "expected customers %s and films %s. %s." % (_p4_matrix_text(expected_customers),
                                                              _p4_matrix_text(expected_films), why)
        if (not _p4_matrix(after_c) or not _p4_matrix(after_f) or len(after_c) != len(customers)
                or len(after_f) != len(films) or len(after_c[0]) != len(customers[0])
                or len(after_f[0]) != len(films[0])):
            return ("%s left the matrices in another shape, customers %s and films %s; %s" % (
                call, _p4_short(after_c), _p4_short(after_f), wanted),
                "move each number in its place, customers[i][position] = customers[i][position] - alpha * "
                "customer_slopes[i][position]: the matrices keep their shape.")
        if not _p4_close_matrix(after_c, expected_customers) or not _p4_close_matrix(after_f, expected_films):
            got = "%s left customers at %s and films at %s; %s" % (call, _p4_matrix_text(after_c),
                                                                    _p4_matrix_text(after_f), wanted)
            if _p4_close_matrix(after_c, customers) and _p4_close_matrix(after_f, films):
                return (got, "no number moved in the lists that the function got. In the last loops, set "
                             "customers[i][position] = customers[i][position] - alpha * customer_slopes[i][position], "
                             "and the same for films[position][j] with film_slopes[position][j]. A line customers = "
                             "... or films = ... inside the function makes a new list, and the old one stays as it "
                             "was.")
            for how, hint in _P4_STEP_SLIPS:
                same = True
                for test in _P4_TESTS["table_step"]:
                    a, e, c = _p4_call(fn, test[0])
                    if e is not None:
                        same = False
                        break
                    vc, vf = _p4_step_variant(test[0][0], test[0][1], test[0][2], test[0][3], how)
                    if not _p4_close_matrix(c[0], vc) or not _p4_close_matrix(c[1], vf):
                        same = False
                        break
                if same:
                    return (got, hint)
            near = True
            for mine, right in ((after_c, expected_customers), (after_f, expected_films)):
                for i in range(len(right)):
                    for k in range(len(right[0])):
                        if abs(mine[i][k] - right[i][k]) > 0.05 * max(1.0, abs(right[i][k])):
                            near = False
            if near:
                return (got, "close, but off. Measure every slope with customer_slope and film_slope, all of them "
                             "before any number moves, then move each number by alpha times its slope.")
            return (got, "first customer_slopes and film_slopes, filled with customer_slope(customers, films, "
                         "ratings, i, position) and film_slope(customers, films, ratings, j, position), each slope "
                         "in the place of its number; then customers[i][position] = customers[i][position] - alpha "
                         "* customer_slopes[i][position], and films[position][j] = films[position][j] - alpha * "
                         "film_slopes[position][j].")
        if answer is None:
            return ("%s moved the numbers right but gave no answer (None)" % call,
                    "end the function with return average_loss(customers, films, ratings), the loss after the step.")
        if not _p4_close(answer, expected_loss):
            if _p4_close(answer, _p4_loss(customers, films, ratings)):
                return ("%s moved the numbers right but returned %s, the loss before the step; expected %s"
                        % (call, _p4_dec(answer, 6), _p4_dec(expected_loss, 6)),
                        "return the loss after the step: average_loss(customers, films, ratings) on the last line, "
                        "after the numbers have moved.")
            return ("%s moved the numbers right but returned %s; expected the average loss after the step, %s"
                    % (call, _p4_short(answer), _p4_dec(expected_loss, 6)),
                    "end the function with return average_loss(customers, films, ratings), the loss after the step.")
    return None


# Task E, factorize

def _p4_call_recorded(fn, inputs, calls):
    """Run factorize with table_step replaced by a recorder, which makes the step right and notes the call (the
    ratings and alpha it got). Gives back what _p4_call gives back."""
    space = getattr(fn, "__globals__", None)
    if not isinstance(space, dict):
        return _p4_call(fn, inputs)
    had = "table_step" in space
    saved = space.get("table_step")

    def recorder(customers, films, ratings, alpha):
        if len(calls) >= _P4_MAX_CALLS:
            raise _P4_Stop("many")
        if (not _p4_matrix(customers) or not _p4_matrix(films) or not _p4_matrix(ratings)
                or not _p4_is_number(alpha) or len(ratings) != len(customers) or len(ratings[0]) != len(films[0])
                or len(customers[0]) != len(films)):
            raise _P4_Stop("call")
        calls.append((_p4_copy(ratings), alpha))
        return _p4_step(customers, films, ratings, alpha)

    space["table_step"] = recorder
    try:
        return _p4_call(fn, inputs)
    finally:
        if had:
            space["table_step"] = saved
        else:
            del space["table_step"]


def _p4_run_factorize(fn):
    """None when factorize passes every test, else (what went wrong, hint)."""
    stop_hint = ("stop when the loss is below 0.0001: at the top of each round of the loop, if loss < 0.0001: "
                 "return losses. And loss must hold the loss after the latest step: loss = table_step(customers, "
                 "films, ratings, alpha).")
    for inputs, expected, call, why in _P4_TESTS["factorize"]:
        customers, films, ratings, alpha, max_steps = inputs
        expected_losses, expected_customers, expected_films = expected
        calls = []
        answer, error, copies = _p4_call_recorded(fn, inputs, calls)
        if error is not None:
            if error == "STOP many":
                return ("%s made more than %d steps and was stopped. %s." % (call, _P4_MAX_CALLS, why),
                        "the loop makes at most max_steps steps, for step_number in range(max_steps), and stops "
                        "early when the loss is below 0.0001.")
            if error == "STOP call":
                return ("%s called table_step with something other than the two matrices, the ratings and a step "
                        "size" % call, "pass the four arguments in this order: table_step(customers, films, "
                        "ratings, alpha).")
            return ("%s stopped with an error: %s" % (call, error), _p4_error_hint(error, "factorize"))
        for used_ratings, used_alpha in calls:
            if used_alpha != alpha:
                return ("%s called table_step with the step size %s; the function got %s"
                        % (call, _p4_short(used_alpha), _p4_short(alpha)),
                        "use alpha, the step size that factorize gets: table_step(customers, films, ratings, "
                        "alpha). The check also tries other step sizes.")
            if used_ratings != ratings:
                return ("%s called table_step with other ratings than the ones it got" % call,
                        "use ratings, the table that factorize gets: table_step(customers, films, ratings, alpha).")
        if len(calls) > max_steps:
            return ("%s made %d steps, and max_steps was %d. %s." % (call, len(calls), max_steps, why),
                    "the given loop, for step_number in range(max_steps), makes at most max_steps steps: keep it "
                    "as it is.")
        if answer is None:
            return ("%s gave no answer (None). %s." % (call, why),
                    "return the list: return losses, both inside the stop and on the last line.")
        if not isinstance(answer, list):
            if _p4_is_number(answer):
                given = "the number %s" % _p4_dec(answer, 6)
            else:
                given = _p4_short(answer)
            return ("%s gave %s; expected the list of losses, %s. %s." % (call, given,
                                                                        _p4_list_text(expected_losses), why),
                    "return losses, the whole list, the loss at the start first.")
        for x in answer:
            if not _p4_is_number(x):
                return ("%s gave a list that holds %s; expected the losses, %s" % (call, _p4_short(x),
                                                                                  _p4_list_text(expected_losses)),
                        "each entry of losses is one number, an average loss: losses.append(loss), with loss = "
                        "table_step(customers, films, ratings, alpha).")
        if _p4_close_list(answer, expected_losses):
            if not _p4_close_matrix(copies[0], expected_customers) or not _p4_close_matrix(copies[1],
                                                                                           expected_films):
                return ("%s gave the right losses, but left customers at %s and films at %s"
                        % (call, _p4_matrix_text(copies[0]), _p4_matrix_text(copies[1])),
                        "factorize changes the two matrices itself, through table_step(customers, films, ratings, "
                        "alpha): work on customers and films, the lists that the function gets.")
            continue
        got = "%s gave %s; expected %s. %s." % (call, _p4_list_text(answer), _p4_list_text(expected_losses), why)
        n, n_expected = len(answer), len(expected_losses)
        if not calls and n_expected > 1:
            if _p4_uses(fn, "table_step"):
                return (got, "no step was made: the function returned before its first step. Return early only "
                             "when the loss is below 0.0001, if loss < 0.0001: return losses, and make the step "
                             "after that line.")
            return (got, "no step was made. In the loop, loss = table_step(customers, films, ratings, alpha) makes "
                         "one step and gives back the average loss after it.")
        if n == n_expected - 1 and _p4_close_list(answer, expected_losses[1:]):
            return (got, "the loss at the start is missing: the list begins with it, losses = [loss], before the "
                         "loop.")
        if n == 1 and n_expected > 1 and _p4_close(answer[0], expected_losses[0]):
            return (got, "losses holds only the loss at the start. Put each new loss at the end of the list, "
                         "losses.append(loss), right after the step.")
        if n > n_expected and _p4_close_list(answer[:n_expected], expected_losses):
            if n_expected == 1:
                return ("%s made a step from a guess that needs none: the list holds %d losses, and 1 was expected. "
                        "%s." % (call, n, why),
                        "the stop comes first in each round, before the step, so a guess that is already good "
                        "makes no step: if loss < 0.0001: return losses, then loss = table_step(...).")
            return ("%s went on after the loss was below 0.0001: the list holds %d losses, and %d were expected. "
                    "%s." % (call, n, n_expected, why), stop_hint)
        repeats = True
        for x in answer:
            if not _p4_close(x, answer[0]):
                repeats = False
        if n > 1 and repeats and not _p4_close_list(answer, expected_losses[:n]):
            return (got, "the list repeats the loss at the start: loss = table_step(customers, films, ratings, "
                         "alpha) keeps the loss after each step, and losses.append(loss) puts it in the list.")
        if n < n_expected and _p4_close_list(answer, expected_losses[:n]):
            return ("%s stopped after %s, with the loss at %s, above 0.0001. %s."
                    % (call, _p4_count(n - 1, "step"), _p4_loss_text(answer[-1]), why),
                    "return early only when the loss is below 0.0001: if loss < 0.0001: return losses. "
                    "Otherwise the loop goes on, and the last line, return losses, comes after max_steps steps.")
        return (got, "losses = [loss] before the loop; in each round, first the stop, then loss = "
                     "table_step(customers, films, ratings, alpha), then losses.append(loss).")
    return None


_P4_RUNNERS = {"average_loss": _p4_run_average_loss,
               "customer_slope": lambda fn: _p4_run_slope(fn, "customer_slope"),
               "film_slope": lambda fn: _p4_run_slope(fn, "film_slope"),
               "table_step": _p4_run_table_step, "factorize": _p4_run_factorize}
_P4_NEEDS = {"average_loss": [("inner_product", ""), ("column", "")],
             "customer_slope": [("inner_product", ""), ("column", "")],
             "film_slope": [("inner_product", ""), ("column", "")],
             "table_step": [("customer_slope", "task B"), ("film_slope", "task C"), ("average_loss", "task A")],
             "factorize": [("table_step", "task D"), ("average_loss", "task A")]}
_P4_TASK = {"average_loss": "A", "customer_slope": "B", "film_slope": "C", "table_step": "D", "factorize": "E"}


def _p4_works(name):
    """True when the function called name exists and passes its tests."""
    fn = globals().get(name)
    if not callable(fn):
        return False
    if name == "inner_product":
        for inputs, wanted in ((([2, 1], [1, 2]), 4), (([-1, -2], [-1, 2]), -3), (([3], [-2]), -6),
                               (([0.5, 1, 2], [2, 0.5, -1]), -0.5)):
            answer, error, after = _p4_call(fn, inputs)
            if error is not None or not _p4_close(answer, wanted):
                return False
        return True
    if name == "column":
        for inputs, wanted in ((([[1, 2], [3, 4], [5, 6]], 1), [2, 4, 6]), (([[-1], [2]], 0), [-1, 2]),
                               (([[-1, -2], [2, -1]], 1), [-2, -1])):
            answer, error, after = _p4_call(fn, inputs)
            if error is not None or answer != wanted:
                return False
        return True
    if _p4_has_gap(fn):
        return False
    return _P4_RUNNERS[name](fn) is None


def _p4_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        if _p4_is_number(fn):
            got = "the number %s" % _p4_dec(fn, 6)
        else:
            got = _p4_short(fn)
        print("❌ check_%s needs the function %s itself, and it got %s. Write check_%s(%s), with no brackets "
              "after %s. If a cell set %s = ..., run the cell of task %s again to bring the function back."
              % (name, name, got, name, name, name, name, _P4_TASK[name]))
        return
    if _p4_has_gap(fn):
        print("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, run its "
              "cell, then run this check again." % name)
        return
    for other, place in _P4_NEEDS[name]:
        if not _p4_uses(fn, other) or _p4_works(other):
            continue
        if other in ("inner_product", "column"):
            print("⏳ %s uses %s, which is missing or was changed. Run the cell with part 1's functions at the top "
                  "of part 4 again (they are given, so undo any change to them), then run this check again."
                  % (name, other))
        else:
            print("⏳ %s uses your %s from %s, which does not work yet. Get the check of %s to ✅ first, then run "
                  "this check again." % (name, other, place, place))
        return
    failure = _P4_RUNNERS[name](fn)
    if failure is None:
        print("✅ %s works on %d tests. %s" % (name, len(_P4_TESTS[name]), _P4_SUCCESS[name]))
        return
    print("❌ " + failure[0])
    print("   💡 Hint: " + failure[1])


def check_average_loss(fn):
    """Task A: tests average_loss on the column, the whole table, part 1's scores, one rating, an exact guess
    and a guess with decimals."""
    try:
        _p4_check("average_loss", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 4 again, then this cell." % error)


def check_customer_slope(fn):
    """Task B: tests customer_slope on every customer's number of the column and of the whole table, and on
    part 1's scores and decimals."""
    try:
        _p4_check("customer_slope", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 4 again, then this cell." % error)


def check_film_slope(fn):
    """Task C: tests film_slope on every film's number of the column and of the whole table, and on part 1's
    scores and decimals."""
    try:
        _p4_check("film_slope", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 4 again, then this cell." % error)


def check_table_step(fn):
    """Task D: tests table_step on six guesses with two step sizes: the matrices after it and the loss."""
    try:
        _p4_check("table_step", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 4 again, then this cell." % error)


def check_factorize(fn):
    """Task E: tests factorize on six runs: the list of losses, the stop, max_steps, the matrices at the end."""
    try:
        _p4_check("factorize", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 4 again, then this cell." % error)


print("✅ Setup of part 4 complete.")

## Part 1's functions, given again

Part 4 multiplies rows and columns. Here are part 1's `inner_product`, `column`, `scores_for_customer` and
`matrix_multiply`, as part 1 writes them, given again so that this part works even if your part 1 is not
finished. `column(films, j)` gives the numbers of film `j`, the column `j` of `films`, from the top down. Run the
cell.

In [ ]:
# Given again: part 1's inner_product, column, scores_for_customer and matrix_multiply, as part 1 writes
# them, so that part 4 works on its own, even when part 1 is not finished
def inner_product(row_a, row_b):
    # row_a and row_b are two lists of numbers of the same length,
    # such as a customer's row and a film's column.
    # Multiply them position by position, then add up the products.
    total = 0
    for i in range(len(row_a)):
        total = total + row_a[i] * row_b[i]
    return total


# Given: column j of a matrix, as a list
def column(matrix, j):
    # matrix is a list of rows. Column j holds the number at position j of every row, from the top down.
    numbers = []
    for row in matrix:
        numbers.append(row[j])
    return numbers


def scores_for_customer(customer_row, film_matrix):
    # customer_row is the row of one customer; film_matrix has one column per film.
    # Return a list with the customer's score for every film, in the order of the columns.
    scores = []
    for j in range(len(film_matrix[0])):       # one round per column: one per film
        film_column = column(film_matrix, j)   # the column of film number j
        score = inner_product(customer_row, film_column)
        scores.append(score)
    return scores


def matrix_multiply(matrix_a, matrix_b):
    # The columns of matrix_a match the rows of matrix_b. The product has one row for each row
    # of matrix_a and one column for each column of matrix_b: its row i holds the inner
    # products of matrix_a[i] with every column of matrix_b.
    product = []
    for i in range(len(matrix_a)):
        # Row i of the product: the scores of matrix_a[i] with every column of matrix_b (part 1's task B)
        new_row = scores_for_customer(matrix_a[i], matrix_b)
        product.append(new_row)
    return product

## The ratings and the first guess

Part 2's table of ratings, written again: Ida, Raj and Elsa rated *Paddington* and *The Martian*, one row per customer and one column per
film. The customers and their ratings are invented for the lesson.

In [ ]:
# The ratings of part two of the game (invented for the lesson): one row per customer, one column per film
customer_names2 = ["Ida", "Raj", "Elsa"]
film_names2 = ["Paddington", "The Martian"]
ratings = [
    [ 2,  1],   # Ida
    [-1,  2],   # Raj
    [ 1, -2],   # Elsa
]

The Matchmaker's first guess of part 2, as two matrices again, as on the board of the game: `customers` has one
row per customer, and `films` one column per film, its first number on top. Each customer and each film has two
numbers, and nobody knows what they mean, so they have no names.

In [ ]:
# The Matchmaker's first guess of part 2, as two matrices: two numbers for every customer and every film
customers = [
    [-1, -2],   # Ida
    [-2,  0],   # Raj
    [ 2,  2],   # Elsa
]
# The films: one row per number, one column per film, in the order of film_names2: Paddington, The Martian
films = [
    [-1, -2],   # first number
    [ 2, -1],   # second number
]
number_names = ["first number", "second number"]
show_matrix(customers, customer_names2, number_names)
show_matrix(films, number_names, film_names2)

The prediction of a customer for a film is the inner product of the customer's row and the film's column. Every
customer's row with every film's column: that is the product of the two matrices, part 1's matrix
multiplication.

In [ ]:
# The predictions of the guess: every customer's row with every film's column, the product of the matrices
show_matrix(matrix_multiply(customers, films), customer_names2, film_names2)

**Reading the result.** Ida's predictions are -3 and 4, against the ratings 2 and 1; Raj's predictions are 2 and 4, against the ratings -1 and 2; Elsa's predictions are 2 and -6, against the ratings 1 and -2. The first guess misses every rating.

## The column

As in the game, we start with the three customers and one film, *Paddington*: the first column of the table.
`column_films` holds Paddington's two numbers of the first guess, as a matrix of one column, and `column_ratings` the
column of ratings. The cell below prints every rating of the column with its prediction and its loss, the
prediction minus the rating, squared.

In [ ]:
# The column of the game: the three customers and one film, Paddington
column_films = [[-1], [2]]                   # Paddington's column of the first guess, its two numbers
column_film_names = ["Paddington"]
column_ratings = [
    [ 2],   # Ida
    [-1],   # Raj
    [ 1],   # Elsa
]
show_guess(customers, column_films, column_ratings, customer_names2, column_film_names)

**Reading the result.** Three predictions, -3, 2 and 2, against the ratings 2, -1 and 1, and three losses, 25, 9 and 1. The Matchmaker
wants one number for the whole column: the **average** of the losses, (25 + 9 + 1) / 3, about 11.67. An average stays
comparable when more customers or films join.

## 🎯 Task A. The average loss

Write `average_loss(customers, films, ratings)`. For every customer `i` and every film `j`: the prediction
`inner_product(customers[i], column(films, j))`, the difference, the prediction minus `ratings[i][j]`, and its
square, the loss of that rating. Then the average of these losses over all the ratings. Two loops, one inside
the other, visit every rating, the way part 1's `matrix_multiply` visited every cell: `i` runs over the rows of
`customers` and `j` over the columns of `films`. On the column it gives about 11.6667, and on the whole table about
10.6667.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# The two loops visit every rating: i runs over the customers, j over the films, the columns
# of films. For each rating: the prediction, the difference prediction - ratings[i][j], and
# its square, difference * difference, which goes into total. count goes up by 1.
# After both loops, total holds the sum of the losses and count the number of ratings.

In [ ]:
#@title 💡 Hint 2 for task A: the shape of the code (double-click to open) { display-mode: "form" }
#             prediction = inner_product(customers[i], column(films, j))
#             difference = prediction - ratings[i][j]
#             total = total + ... * ...
#             count = count + 1
#     return ... / ...                             # after both loops, at the level of total = 0

In [ ]:
# ✅ SOLUTION, task A: the average loss of a table
def average_loss(customers, films, ratings):
    # The loss of a whole table. films has one row per number and one column per film. For every
    # customer i and every film j: the prediction is the inner product of the customer's row and
    # the film's column, and the loss of that rating is the prediction minus the rating, squared.
    # Return the average of these losses over all the ratings.
    total = 0                                      # the sum of the losses
    count = 0                                      # the number of ratings
    for i in range(len(customers)):
        for j in range(len(films[0])):             # one round per film: the columns of films
            prediction = inner_product(customers[i], column(films, j))
            difference = prediction - ratings[i][j]
            total = total + difference * difference    # the loss of this rating
            count = count + 1
    return total / count                           # the average of the losses

In [ ]:
check_average_loss(average_loss)

In [ ]:
# The average loss of the column, then the whole table rating by rating, and its average loss
print_loss("The average loss of the column", average_loss(customers, column_films, column_ratings))
show_guess(customers, films, ratings, customer_names2, film_names2)
print_loss("The average loss of the table", average_loss(customers, films, ratings))

**Reading the result.** The column's average is 35 / 3, about 11.67. The whole table adds *The Martian*: six losses, 25, 9, 9, 4, 1 and 16,
whose sum is 64 and whose average is 64 / 6, about 10.67. The same function works for a table of any size.

## Every rating has a say

In the game, a number that sits in one prediction has the slope 2 times its **partner**, the number it meets
in that prediction, times the **difference**, the prediction minus the rating. Paddington's first number sits in three predictions,
one for each customer of the column. So each of the three ratings, alone, gives it such a slope.

💭 **Think first.** Do the three ratings agree? Then run the cell below.

In [ ]:
# What each rating of the column, alone, gives Paddington's first number, column_films[0][0]:
# 2 times its partner, the customer's first number, times the difference, the prediction minus the rating
total = 0
for i in range(len(customers)):
    prediction = inner_product(customers[i], column(column_films, 0))
    difference = prediction - column_ratings[i][0]
    partner = customers[i][0]
    from_this_rating = 2 * partner * difference
    print(customer_names2[i] + "'s rating gives it", from_this_rating)
    total = total + from_this_rating
print("Their average over the 3 ratings, the slope:", round(total / 3, 4))

**Reading the result.** Ida's rating gives 10 and Elsa's 4: they say "smaller". Raj's gives -12: it says "bigger".
The slope of the average loss is the average of what the ratings give: (10 - 12 + 4) / 3 = 2 / 3, about 0.67. A small
positive slope: Paddington's first number should shrink a little.

## 🎯 Task B. The slope of a customer's number

A customer's number sits in fewer predictions. Ida's first number sits only in Ida's prediction for *Paddington*: her rating gives it
10, which is 2 times its partner, -1, times the difference, -5. The ratings of Raj and Elsa give it 0, because their
predictions do not contain it. The slope of the average loss is again the average over all 3 ratings, (10 + 0 + 0) /
3, about 3.33: the ratings that give 0 still count.

So for a customer's number `customers[i][position]`: loop over the films `j`, add up 2 times the partner
`films[position][j]`, film `j`'s number at the same position, times the difference `prediction - ratings[i][j]`,
and divide by the number of **all** the ratings of the table, `len(customers) * len(films[0])`: `films` has one
column per film.

Write `customer_slope(customers, films, ratings, i, position)`. On the column,
`customer_slope(customers, column_films, column_ratings, 0, 0)` gives about 3.3333.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The loop visits the ratings of customer i, one for every film j, a column of films.
# For each: the difference, prediction - ratings[i][j], and the partner, films[position][j],
# film j's number at the same position, the number that customers[i][position] meets in this
# prediction. This rating gives the number 2 * partner * difference: add it to total.

In [ ]:
#@title 💡 Hint 2 for task B (double-click to open) { display-mode: "form" }
# count is the number of ALL the ratings of the table: len(customers) * len(films[0]), since
# films has one column per film. The ratings of the other customers give the number 0: they
# add nothing to total, and they still count. On the column that is 3 ratings, so
# Ida's first number gets (10 + 0 + 0) / 3.

In [ ]:
#@title 💡 Hint 3 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#         difference = prediction - ratings[i][j]
#         partner = films[position][j]
#         total = total + 2 * ... * ...
#     count = len(...) * len(...)
#     return total / count

In [ ]:
# ✅ SOLUTION, task B: the slope of a customer's number
def customer_slope(customers, films, ratings, i, position):
    # The slope of the average loss for one customer's number, customers[i][position]. It sits in
    # the predictions of customer i, one for every film j. Each of those ratings alone gives it 2
    # times its partner times the difference, the prediction minus the rating; every other rating
    # gives it 0. Add up what the ratings give, and divide by the number of ALL the ratings.
    total = 0                                      # what the ratings give the number, added up
    for j in range(len(films[0])):                 # one round per film: the columns of films
        prediction = inner_product(customers[i], column(films, j))
        difference = prediction - ratings[i][j]
        partner = films[position][j]               # the number it meets in this prediction
        total = total + 2 * partner * difference   # what this rating gives the number
    count = len(customers) * len(films[0])         # ALL the ratings of the table
    return total / count

In [ ]:
check_customer_slope(customer_slope)

## 🎯 Task C. The slope of a film's number

The mirror image. A film's number `films[position][j]` sits in the predictions of every customer for film `j`:
loop over the customers `i`, add up 2 times the partner `customers[i][position]` times the difference, and
divide by the number of all the ratings of the table. On the column, Paddington's first number gets (10 - 12 + 4) / 3, about 0.6667, as the cell
above found, and Paddington's second number (20 + 0 + 4) / 3 = 8.

Write `film_slope(customers, films, ratings, j, position)`.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# The same as task B with the roles swapped: the loop visits the ratings of film j, one for
# every customer i, and the partner of films[position][j] is customers[i][position].
# count is still the number of ALL the ratings: len(customers) * len(films[0]).

In [ ]:
#@title 💡 Hint 2 for task C: the shape of the code (double-click to open) { display-mode: "form" }
#         difference = prediction - ratings[i][j]
#         partner = customers[i][position]
#         total = total + 2 * ... * ...
#     count = len(...) * len(...)
#     return total / count

In [ ]:
# ✅ SOLUTION, task C: the slope of a film's number
def film_slope(customers, films, ratings, j, position):
    # The slope of the average loss for one film's number, films[position][j]: film j's number at
    # this position, in row position and column j. It sits in the predictions of film j, one for
    # every customer i. Each of those ratings alone gives it 2 times its partner times the
    # difference, the prediction minus the rating; every other rating gives it 0. Add up what the
    # ratings give, and divide by the number of ALL the ratings of the table.
    total = 0                                      # what the ratings give the number, added up
    for i in range(len(customers)):
        prediction = inner_product(customers[i], column(films, j))
        difference = prediction - ratings[i][j]
        partner = customers[i][position]           # the number it meets in this prediction
        total = total + 2 * partner * difference   # what this rating gives the number
    count = len(customers) * len(films[0])         # ALL the ratings of the table
    return total / count

In [ ]:
check_film_slope(film_slope)

In [ ]:
# Every slope on the column, then on the whole table, from the first guess
print("On the column")
show_slopes(customers, column_films, column_ratings, customer_names2, column_film_names)
print("On the whole table")
show_slopes(customers, films, ratings, customer_names2, film_names2)

**Reading the result.** On the column the eight slopes of the game: Ida 3.33 and -6.67; Raj -2 and 4; Elsa -0.67 and 1.33; *Paddington* 0.67 and 8. On the whole table a customer's
number sits in two predictions and a film's number in three, and the average runs over all six ratings: Ida -0.33 and -4.33; Raj -2.33 and 1.33; Elsa 2.33 and 2;
*Paddington* 0.33 and 4, *The Martian* -5 and -4.67. A negative slope says that the number should grow, a positive one that it should shrink.

## 🎯 Task D. Everyone moves at once

As in the game, every number moves against its slope: **the new number is the old number minus alpha times its
slope**, all the numbers in one step. Every slope is measured on the same guess, before any number moves. So the
function first fills two matrices of slopes, `customer_slopes` and `film_slopes`, with your functions of tasks B
and C, each slope in the place of its number (`film_slopes` has one row per number and one column per film, as
`films`), and only then moves the numbers.

Here the step size is alpha = 0.2, ten times part 3's 0.02: the average divides every slope by the number
of ratings, so the slopes are smaller and the steps can be bigger. With 0.02 the whole table would need 597 steps,
and with 0.2 it needs 52.

Write `table_step(customers, films, ratings, alpha)`. It changes `customers` and `films` themselves and returns
the average loss after the step. On the column, from the first guess, the average falls from about 11.6667 to 4.594.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# The first loops only measure: each slope comes from your customer_slope (task B) or your
# film_slope (task C), called with the whole guess and the place of the number, i or j and
# position (film_slope takes the film first). No number moves in them, so every slope
# belongs to the same guess.
# The last loops move: each number takes alpha times its slope away from itself.

In [ ]:
#@title 💡 Hint 2 for task D (double-click to open) { display-mode: "form" }
# For Ida's first number on the column: -1 - 0.2 * 3.33 = -1.67.
# In Python: customers[i][position] = customers[i][position] - alpha * customer_slopes[i][position]
# and the same for films[position][j] with film_slopes[position][j].

In [ ]:
#@title 💡 Hint 3 for task D: the shape of the code (double-click to open) { display-mode: "form" }
#             row_of_slopes.append(customer_slope(customers, films, ratings, i, position))
#     ...
#             row_of_slopes.append(film_slope(customers, films, ratings, j, position))
#     ...
#             customers[i][position] = customers[i][position] - ... * ...
#     ...
#             films[position][j] = films[position][j] - ... * ...

In [ ]:
# ✅ SOLUTION, task D: one step for every number at once
def table_step(customers, films, ratings, alpha):
    # One step for every number of both matrices. First measure every slope on the guess as it is
    # now; then move every number against its slope: the old number minus alpha times its slope.
    # Return the average loss after the step.
    # First: every slope, before any number moves, each in the place of its number
    customer_slopes = []
    for i in range(len(customers)):
        row_of_slopes = []
        for position in range(len(customers[i])):
            row_of_slopes.append(customer_slope(customers, films, ratings, i, position))
        customer_slopes.append(row_of_slopes)
    film_slopes = []                               # the shape of films: one row per number
    for position in range(len(films)):
        row_of_slopes = []
        for j in range(len(films[position])):      # one column per film
            row_of_slopes.append(film_slope(customers, films, ratings, j, position))
        film_slopes.append(row_of_slopes)
    # Then: every number moves against its slope
    for i in range(len(customers)):
        for position in range(len(customers[i])):
            customers[i][position] = customers[i][position] - alpha * customer_slopes[i][position]
    for position in range(len(films)):
        for j in range(len(films[position])):
            films[position][j] = films[position][j] - alpha * film_slopes[position][j]
    return average_loss(customers, films, ratings)     # the average loss after the step

In [ ]:
check_table_step(table_step)

In [ ]:
# One step on the column, from the first guess, with the step size of the game
alpha = 0.2                                  # part 3 used 0.02
customers = [[-1, -2], [-2, 0], [2, 2]]      # the first guess again: Ida, Raj and Elsa
column_films = [[-1], [2]]                   # Paddington's column
print("Before the step")
show_guess(customers, column_films, column_ratings, customer_names2, column_film_names)
loss_after = table_step(customers, column_films, column_ratings, alpha)
print("After the step")
show_guess(customers, column_films, column_ratings, customer_names2, column_film_names)
print_loss("The average loss after the step", loss_after)

**Reading the result.** One step, and the average fell from 11.67 to 4.59. Ida's loss fell from 25 to 0.14 and Raj's from
9 to 6.22, and Elsa's rose from 1 to 7.42.

💭 Why is it fine that Elsa's loss rose? Think first, then open the folded answer below.

In [ ]:
#@title 💭 Answer: why Elsa's loss may rise (double-click to open) { display-mode: "form" }
# Elsa's prediction depends on her two numbers and on Paddington's two numbers. Paddington's numbers
# serve all three ratings at once, and they moved the way the average of the three asked,
# which took Elsa's prediction away from her rating for one step.
# The step makes the AVERAGE loss smaller: here from 11.67 to 4.59. Step after step all three
# losses get small: after 13 steps every loss of the column is about 0.

In [ ]:
# One step on the whole table, from the first guess
customers = [[-1, -2], [-2, 0], [2, 2]]      # the first guess again: Ida, Raj and Elsa
films = [[-1, -2], [2, -1]]                  # the columns: Paddington and The Martian
print_loss("The average loss before the step", average_loss(customers, films, ratings))
loss_after = table_step(customers, films, ratings, alpha)
print_loss("The average loss after the step", loss_after)
show_guess(customers, films, ratings, customer_names2, film_names2)

**Reading the result.** On the whole table one step takes the average from 10.67 to 1.97: all ten numbers moved at
once, six for the customers and four for the films.

## 🎯 Task E. Again and again: factorize

The loop repeats the step: measure every slope, move every number, and go again. Below 0.0001 the average loss
counts as 0. Write `factorize(customers, films, ratings, alpha, max_steps)`. It makes at most `max_steps` steps,
with `for step_number in range(max_steps)`, and stops early, with `return`, once the average loss is below 0.0001.
It changes `customers` and `films` themselves and returns the list of average losses, the loss at the start
first. A given line, `print_progress`, prints the average loss after steps 1, 2 and 3, then every 10 steps
(every 100 after step 100). If you did part 3, this is its `gradient_descent` for a whole table.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task E (double-click to open) { display-mode: "form" }
# losses starts as a list that holds one number, the loss at the start: losses = [loss].
# Each round of the loop does three things: first the stop, then one step, then the new loss
# goes at the end of the list.

In [ ]:
#@title 💡 Hint 2 for task E (double-click to open) { display-mode: "form" }
# The stop: if loss < 0.0001: return losses. It comes first in the round, so a guess that is
# already exact makes no step at all.
# One step: loss = table_step(customers, films, ratings, alpha) moves every number and gives
# back the average loss after the step.

In [ ]:
#@title 💡 Hint 3 for task E: the shape of the code (double-click to open) { display-mode: "form" }
#     losses = [loss]
#     for step_number in range(max_steps):
#         if loss < 0.0001:
#             return losses
#         loss = table_step(customers, films, ratings, alpha)
#         losses.append(...)
#         print_progress(step_number + 1, loss)
#     return losses

In [ ]:
# ✅ SOLUTION, task E: factorize, step after step
def factorize(customers, films, ratings, alpha, max_steps):
    # table_step again and again. Stop when the average loss is below 0.0001, where it counts as 0,
    # or after max_steps steps. Return the list of average losses, the loss at the start first.
    loss = average_loss(customers, films, ratings)     # the average loss at the start
    losses = [loss]                                # the list of losses, the start first
    for step_number in range(max_steps):
        # Small enough: the loss counts as 0, so stop
        if loss < 0.0001:
            return losses
        loss = table_step(customers, films, ratings, alpha)   # one step, and the loss after it
        losses.append(loss)
        print_progress(step_number + 1, loss)
    return losses

In [ ]:
check_factorize(factorize)

In [ ]:
# factorize on the whole table, from the first guess, at most 1000 steps
customers = [[-1, -2], [-2, 0], [2, 2]]      # the first guess again: Ida, Raj and Elsa
films = [[-1, -2], [2, -1]]                  # the columns: Paddington and The Martian
losses = factorize(customers, films, ratings, alpha, 1000)
show_run(losses)

**Reading the result.** 52 steps, and the average loss fell at every step: 10.6667, 1.9659, 1.3192, 1.0214, 0.8283, ..., 0.00012, 0.0001, about 0. The cell below shows the two matrices
that the loop found, and their product next to the ratings.

In [ ]:
# The two matrices found, and their product next to the ratings
print("The customers' numbers found:")
show_matrix(customers, customer_names2, number_names)
print("The films' numbers found:")
show_matrix(films, number_names, film_names2)
print("Their product, matrix_multiply(customers, films), rounded to 2 decimals:")
show_matrix(matrix_multiply(customers, films), customer_names2, film_names2)
print("The ratings:")
show_matrix(ratings, customer_names2, film_names2)

**Reading the result.** The product of the two matrices found gives every rating within 0.02. The table of
ratings is now the product of two matrices, the customers' numbers times the films' numbers. Splitting a table
into such a product is called **matrix factorization**, and the customers' rows and the films' columns found are
the **embeddings** of the customers and the films, as in part 1. Nobody told the computer what the numbers mean:
it found them from the ratings alone.

## Part 1's scores, from the scores alone

In part 1, `matrix_multiply` computed the table of scores from the profiles of Sara, Leo and Maya and the critic's
answers, three numbers each. Now give `factorize` only the scores, as if they were ratings, with a first guess
of three numbers for every customer and every film, drawn at random. Does it find two matrices whose product
gives the scores?

In [ ]:
# Part 1's scores, the product of the profiles and the critic's answers, as if they were ratings
customer_names = ["Sara", "Leo", "Maya"]
film_names = ["The Intouchables", "Apollo 13", "Men in Black", "Interstellar"]
scores = [
    [ 7,  0, -7, -6],   # Sara
    [-4, -6, 10,  0],   # Leo
    [-1,  7, -2,  2],   # Maya
]
# A first guess of three numbers for every customer and every film, drawn at random; score_films has one
# row per number and one column per film, as films
score_customers = [[-2, 0, 2], [2, -2, -1], [2, 2, 2]]
score_films = [[1, 1, -1, -2], [2, 2, -2, -2], [2, 1, 2, 0]]
score_number_names = ["first number", "second number", "third number"]
print_loss("The average loss of the first guess", average_loss(score_customers, score_films, scores))

### 🎯 Your turn

Run your `factorize` on the scores: `score_customers`, `score_films`, `scores`, the step size `alpha`, and at
most 1000 steps.

In [ ]:
# ✅ SOLUTION: factorize the scores, with the step size alpha and at most 1000 steps
losses = factorize(score_customers, score_films, scores, alpha, 1000)
show_run(losses)

In [ ]:
# The product of the matrices found, next to the scores, and the numbers found for the customers
print("Their product, matrix_multiply(score_customers, score_films), rounded to 2 decimals:")
show_matrix(matrix_multiply(score_customers, score_films), customer_names, film_names)
print("The scores:")
show_matrix(scores, customer_names, film_names)
print("The customers' numbers found:")
show_matrix(score_customers, customer_names, score_number_names)

**Reading the result.** 88 steps, and the product gives every score within 0.05. The numbers found differ from the
profiles of part 1: Sara's profile was 2, 1 and -2, and the loop found -2.64, 1.98 and 1.62 for her.

💭 How can other numbers give the same scores? Think first, then open the folded note below.

In [ ]:
#@title 💭 Note: why the numbers found differ from the profiles (double-click to open) { display-mode: "form" }
# The scores fix only the products: every customer's row with every film's column. Many pairs
# of matrices have the same product. For example, double the first number of every customer
# and halve the first number of every film, the top cell of its column: every product stays
# the same, and so does every score.
# So the loop finds one pair among many, and which one depends on the first guess.
# The numbers found have no names either: nothing says that the first one means real or fiction.

## Three experiments

💭 **Think first** before you run each cell below: what will the average loss do?

1. Part 1's scores with **two** numbers for every customer and every film, in place of three.
2. Your own first guess for the table of part 2.
3. A step that is too big: alpha 1 in place of 0.2, on the whole table from the first guess.

In [ ]:
# Experiment 1: part 1's scores with two numbers for every customer and every film, 50 steps
two_customers = [[-2, 0], [2, 2], [-2, -1]]
two_films = [[2, 2, 2, 1], [2, 1, 2, 2]]
losses = factorize(two_customers, two_films, scores, alpha, 50)
show_run(losses)

**Reading the result.** The average loss falls from 59.25 and stops near 0.423: it never gets below 0.0001, however many
steps the loop makes. 💭 Why? Think first, then open the folded answer below.

In [ ]:
#@title 💭 Answer: why two numbers stop near 0.423 (double-click to open) { display-mode: "form" }
# Part 1's scores are the product of two matrices with three numbers for every customer and
# every film, and no pair of matrices with two numbers each gives all twelve of them. The loop
# finds the best that two numbers can do, an average loss of about 0.423, and stays there.
# The number of numbers each that a table needs is called its rank: these scores have rank 3.
# Part 2's table has two columns, so two numbers each were always enough there.

**Experiment 2, your own guess.** Type six numbers for the customers and four for the films into the cell
below, any numbers from -2 to 2, in place of the Matchmaker's first guess, and run the cell. Run it again
whenever you change the numbers.

### 🎯 Your turn

In [ ]:
# ✅ AN EXAMPLE: a guess of our own, 56 steps (yours may differ)
my_customers = [[2, 1],                      # Ida
                [0, -1],                     # Raj
                [1, 2]]                      # Elsa
# The films: one row per number, one column per film, Paddington then The Martian
my_films = [[1, 2],                          # first number
            [-1, 0]]                         # second number
losses = factorize(my_customers, my_films, ratings, alpha, 1000)
show_run(losses)
show_matrix(matrix_multiply(my_customers, my_films), customer_names2, film_names2)

**Reading the result.** From most guesses the loop gets below 0.0001, and the numbers end somewhere else each
time, with the same product: many pairs of matrices give these ratings. If your loss stops above 0.0001, look at
your guess. With every number at 0, every slope is 0, so no number moves and the loss stays at 2.5. With the two
numbers of every customer and of every film equal, all ones for example, they stay equal, so each works as one
number, and the loss stops at 5/6, about 0.8333, the best that one number per customer and per film can do. A few
other guesses end there too. Then try a guess with other numbers.

In [ ]:
# Experiment 3: a step that is too big, alpha = 1, 3 steps on the whole table from the first guess
trial_customers = [[-1, -2], [-2, 0], [2, 2]]
trial_films = [[-1, -2], [2, -1]]
losses = factorize(trial_customers, trial_films, ratings, 1, 3)
show_run(losses)

**Reading the result.** The average loss grows at every step: 10.6667, 18.4362, 95.1598, 110796.5. With alpha 1 every number jumps far past
the bottom, and the next jump starts from a worse guess. With alpha 0.2 the loss falls at every step.

## The end

That is part 4: your `factorize` starts from a guess, measures the average loss of the whole table, moves every
number against its slope, and goes again, until the product of the two matrices gives the ratings. That is the
end of the optional notebook and of the Matchmaker.